# Statistical Analysis: Chat vs. Avatar Across Israel, Germany, and China
## Analysis hierarchy

This notebook reproduces the manuscript-facing statistical analyses and audit outputs from the final analytic sample.


## 1. Set Up the Working Environment

Load the analysis dependencies and companion utility module used throughout the notebook.


In [1]:
# If needed, uncomment and run once:
# %pip install pandas numpy scipy statsmodels matplotlib openpyxl

from pathlib import Path
import sys
import json
import importlib.util
import warnings
import os
import numpy as np
import pandas as pd
from IPython.display import display
import warnings
from statsmodels.tools.sm_exceptions import ValueWarning

warnings.filterwarnings(
    "ignore",
    message=r"covariance of constraints does not have full rank.*",
    category=ValueWarning,
)

SCRIPT_FILE = Path(os.environ.get("ANALYSIS_UTILS_FILE", "holocaust_analysis_utils.py"))
if not SCRIPT_FILE.exists():
    raise FileNotFoundError(
        "Missing holocaust_analysis_utils.py. Place it next to this notebook."
    )

spec = importlib.util.spec_from_file_location("hoa", SCRIPT_FILE.resolve())
if spec is None or spec.loader is None:
    raise ImportError(f"Could not create an import specification for {SCRIPT_FILE}")
hoa = importlib.util.module_from_spec(spec)
sys.modules["hoa"] = hoa
spec.loader.exec_module(hoa)

print("Analysis module loaded successfully:", SCRIPT_FILE)

MIN_GROUP_N = 5
TEMPLATE_THRESHOLD = 0.92


Analysis module loaded successfully: holocaust_analysis_utils.py


### Configuration validation

Confirm the analysis configuration and required constants before loading study data.


In [2]:
# Validate notebook-wide configuration before any analysis runs.
_REQUIRED_CONFIG = {
    "MIN_GROUP_N": MIN_GROUP_N,
    "TEMPLATE_THRESHOLD": TEMPLATE_THRESHOLD,
}

for _name, _value in _REQUIRED_CONFIG.items():
    if _value is None:
        raise ValueError(f"Configuration variable {_name} is not defined.")

if not isinstance(MIN_GROUP_N, int) or MIN_GROUP_N < 2:
    raise ValueError("MIN_GROUP_N must be an integer of at least 2.")

if not (0 < float(TEMPLATE_THRESHOLD) <= 1):
    raise ValueError("TEMPLATE_THRESHOLD must be between 0 and 1.")

print("Notebook configuration validated:")
print(f"  MIN_GROUP_N = {MIN_GROUP_N}")
print(f"  TEMPLATE_THRESHOLD = {TEMPLATE_THRESHOLD}")


Notebook configuration validated:
  MIN_GROUP_N = 5
  TEMPLATE_THRESHOLD = 0.92


## 2. Define File Paths

Resolve the input, output, and auxiliary-data paths used by the analysis.


In [3]:
MAIN_DATA_FILE = Path(os.environ.get("MAIN_DATA_FILE", "data/study_data.csv"))
OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR", "analysis_output"))

required_files = {
    "Main dataset": MAIN_DATA_FILE,
    "Companion analysis utilities": SCRIPT_FILE,
}
for label, path in required_files.items():
    if not path.exists():
        raise FileNotFoundError(f"{label} was not found: {path}")
    print(f"✓ {label}: {path} ({path.stat().st_size / 1024:.1f} KB)")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Final three-group publication analysis")
print("Main data file:", MAIN_DATA_FILE)
print("Output directory:", OUTPUT_DIR)

✓ Main dataset: data/study_data.csv (4937.6 KB)
✓ Companion analysis utilities: holocaust_analysis_utils.py (52.4 KB)
Final three-group publication analysis
Main data file: data/study_data.csv
Output directory: analysis_output


## 3. Load the Study Data

Load the final study export used for the analysis.

In [4]:
study_raw = pd.read_csv(MAIN_DATA_FILE)

print("Main dataset shape:", study_raw.shape)

print("\nColumns in the main dataset:")
print(study_raw.columns.tolist())

display(study_raw.head())

Main dataset shape: (9147, 7)

Columns in the main dataset:
['session_id', 'user_id', 'stage_id', 'field_id', 'value', 'correct_answer', 'completed_at']


,session_id,user_id,stage_id,field_id,value,correct_answer,completed_at
0,e46ca164-d910-4dd4-a6f9-379f49658f77,501709ee-e9dd-4dd0-a16d-127521471879,_assignments,people_tasks_phase,"markel_stage,ben_lesser_stage",NaN,NaN
1,e46ca164-d910-4dd4-a6f9-379f49658f77,501709ee-e9dd-4dd0-a16d-127521471879,_assignments,markel_tasks_picks,markel_avatar_task,NaN,NaN
2,e46ca164-d910-4dd4-a6f9-379f49658f77,501709ee-e9dd-4dd0-a16d-127521471879,_assignments,ben_lesser_tasks_picks,ben_lesser_chat_task,NaN,NaN
3,e46ca164-d910-4dd4-a6f9-379f49658f77,501709ee-e9dd-4dd0-a16d-127521471879,demographics_1,age,28,NaN,2026-04-09T17:07:57.730230
4,e46ca164-d910-4dd4-a6f9-379f49658f77,501709ee-e9dd-4dd0-a16d-127521471879,demographics_1,gender,male,NaN,2026-04-09T17:07:57.730230


## 4. Run Basic Data-Integrity Checks

Check required fields, participant counts, and basic structural integrity before analysis.


In [5]:
hoa.ensure_columns(
    study_raw,
    ["user_id", "session_id", "stage_id", "field_id", "value",
     "correct_answer", "completed_at"],
    "Main dataset",
)

quality_initial = pd.DataFrame({
    "dataset": ["main"],
    "rows": [len(study_raw)],
    "unique_users": [study_raw["user_id"].nunique(dropna=True)],
    "missing_user_id": [study_raw["user_id"].isna().sum()],
    "missing_session_id": [study_raw["session_id"].isna().sum()],
})
display(quality_initial)
print("Required columns are present.")

,dataset,rows,unique_users,missing_user_id,missing_session_id
0,main,9147,146,0,0


Required columns are present.


## 5. Extract Conversation Events and Validate Explicit Research Groups

Parse conversation events and verify the recorded Israel, Germany, and China cohort labels.


In [6]:
messages_unfiltered, conversation_events_unfiltered = hoa.extract_conversation_events(study_raw)

messages_labeled_unfiltered = messages_unfiltered.copy()
messages_labeled_unfiltered["input_method"] = (
    messages_labeled_unfiltered["input_method"]
    .fillna("unknown")
    .astype(str)
    .str.lower()
    .str.strip()
)
messages_labeled_unfiltered["input_method_source"] = np.where(
    messages_labeled_unfiltered["input_method"].isin(["typed", "voice"]),
    "main_event_payload",
    "unknown",
)

study, messages_labeled, country_audit = hoa.assign_research_groups(
    study_raw.copy(), messages_labeled_unfiltered.copy()
)
country_audit_all = country_audit.copy()

conversation_events = conversation_events_unfiltered.merge(
    country_audit[["user_id", "country"]],
    on="user_id", how="left", validate="many_to_one"
)

print("Participants in final dataset:", study["user_id"].nunique())
print("Extracted message participants:", messages_labeled["user_id"].nunique())
print("\nExplicit research-group distribution:")
display(
    country_audit.groupby(["country", "country_source"], dropna=False)
    .agg(n_users=("user_id", "nunique")).reset_index()
)

assert country_audit["user_id"].nunique() == study["user_id"].nunique()
assert set(country_audit["country"]) == {"israel", "germany", "china"}

Participants in final dataset: 146
Extracted message participants: 146

Explicit research-group distribution:


,country,country_source,n_users
0,china,study_country_field,54
1,germany,study_country_field,43
2,israel,study_country_field,49


## 6. Inspect the Extracted Conversation Events

Inspect the extracted participant and agent events before preprocessing and aggregation.


In [7]:
print("Total extracted conversation events:", len(conversation_events))
print(
    "Participant-message events:",
    int(conversation_events["event_kind"].eq("user").sum())
)
print(
    "Agent-response events:",
    int(conversation_events["event_kind"].eq("agent").sum())
)
print("Extracted participant turns:", len(messages_labeled))

display(
    conversation_events[
        ["user_id", "session_id", "stage_id", "country",
         "event_kind", "text", "client_timestamp"]
    ].head(12)
)

display(
    messages_labeled[
        ["user_id", "session_id", "stage_id", "country",
         "user_question_text", "agent_response_text",
         "client_timestamp"]
    ].head(8)
)


print("\nAutomatic opening-pair audit:")
print(
    "Automatic user greetings retained in full audit table:",
    int(conversation_events["is_auto_intro"].sum())
)
print(
    "Agent replies to automatic greetings retained in full audit table:",
    int(conversation_events["is_auto_intro_agent_response"].sum())
)


Total extracted conversation events: 5103
Participant-message events: 2560
Agent-response events: 2543
Extracted participant turns: 2560


,user_id,session_id,stage_id,country,event_kind,text,client_timestamp
0,001b8917-b3bf-420a-bab6-a683e61fbda8,45fc3b5a-cd0c-466e-99ab-8233b3ac089a,ben_lesser_avatar_task,israel,user,"Hello, My name is XXXX XXXX, please tell me ab...",2026-09-03 18:19:17.167000+00:00
1,001b8917-b3bf-420a-bab6-a683e61fbda8,45fc3b5a-cd0c-466e-99ab-8233b3ac089a,ben_lesser_avatar_task,israel,agent,"I am Ben Lesser, a Holocaust survivor born in ...",2026-09-03 18:19:19.441000+00:00
2,001b8917-b3bf-420a-bab6-a683e61fbda8,45fc3b5a-cd0c-466e-99ab-8233b3ac089a,ben_lesser_avatar_task,israel,user,what do you want to tell me about your experience,2026-09-03 18:20:03.817000+00:00
3,001b8917-b3bf-420a-bab6-a683e61fbda8,45fc3b5a-cd0c-466e-99ab-8233b3ac089a,ben_lesser_avatar_task,israel,agent,My life was defined by the loss of my family a...,2026-09-03 18:20:05.773000+00:00
4,001b8917-b3bf-420a-bab6-a683e61fbda8,45fc3b5a-cd0c-466e-99ab-8233b3ac089a,ben_lesser_avatar_task,israel,user,what is important to you that you want to pass...,2026-09-03 18:21:17.273000+00:00
5,001b8917-b3bf-420a-bab6-a683e61fbda8,45fc3b5a-cd0c-466e-99ab-8233b3ac089a,ben_lesser_avatar_task,israel,agent,It is vital that the next generation understan...,2026-09-03 18:21:19.385000+00:00
6,001b8917-b3bf-420a-bab6-a683e61fbda8,45fc3b5a-cd0c-466e-99ab-8233b3ac089a,ben_lesser_avatar_task,israel,user,what do you want us to remember about your family,2026-09-03 18:22:06.263000+00:00
7,001b8917-b3bf-420a-bab6-a683e61fbda8,45fc3b5a-cd0c-466e-99ab-8233b3ac089a,ben_lesser_avatar_task,israel,agent,"My family was a loving, Orthodox unit in Krakó...",2026-09-03 18:22:08.223000+00:00
8,001b8917-b3bf-420a-bab6-a683e61fbda8,45fc3b5a-cd0c-466e-99ab-8233b3ac089a,ben_lesser_avatar_task,israel,user,Scrub your life before everything started.,2026-09-03 18:22:25.517000+00:00
9,001b8917-b3bf-420a-bab6-a683e61fbda8,45fc3b5a-cd0c-466e-99ab-8233b3ac089a,ben_lesser_avatar_task,israel,agent,My early life in Kraków was filled with the wa...,2026-09-03 18:22:27.495000+00:00


,user_id,session_id,stage_id,country,user_question_text,agent_response_text,client_timestamp
0,001b8917-b3bf-420a-bab6-a683e61fbda8,45fc3b5a-cd0c-466e-99ab-8233b3ac089a,ben_lesser_avatar_task,israel,"Hello, My name is XXXX XXXX, please tell me ab...","I am Ben Lesser, a Holocaust survivor born in ...",2026-09-03 18:19:17.167000+00:00
1,001b8917-b3bf-420a-bab6-a683e61fbda8,45fc3b5a-cd0c-466e-99ab-8233b3ac089a,ben_lesser_avatar_task,israel,what do you want to tell me about your experience,My life was defined by the loss of my family a...,2026-09-03 18:20:03.817000+00:00
2,001b8917-b3bf-420a-bab6-a683e61fbda8,45fc3b5a-cd0c-466e-99ab-8233b3ac089a,ben_lesser_avatar_task,israel,what is important to you that you want to pass...,It is vital that the next generation understan...,2026-09-03 18:21:17.273000+00:00
3,001b8917-b3bf-420a-bab6-a683e61fbda8,45fc3b5a-cd0c-466e-99ab-8233b3ac089a,ben_lesser_avatar_task,israel,what do you want us to remember about your family,"My family was a loving, Orthodox unit in Krakó...",2026-09-03 18:22:06.263000+00:00
4,001b8917-b3bf-420a-bab6-a683e61fbda8,45fc3b5a-cd0c-466e-99ab-8233b3ac089a,ben_lesser_avatar_task,israel,Scrub your life before everything started.,My early life in Kraków was filled with the wa...,2026-09-03 18:22:25.517000+00:00
5,001b8917-b3bf-420a-bab6-a683e61fbda8,45fc3b5a-cd0c-466e-99ab-8233b3ac089a,ben_lesser_avatar_task,israel,describe,My early life in Kraków was defined by the war...,2026-09-03 18:22:31.399000+00:00
6,001b8917-b3bf-420a-bab6-a683e61fbda8,45fc3b5a-cd0c-466e-99ab-8233b3ac089a,ben_lesser_avatar_task,israel,What's the first thing that. You remember. Fe...,The first time I truly felt unsafe was when th...,2026-09-03 18:23:12.489000+00:00
7,001b8917-b3bf-420a-bab6-a683e61fbda8,45fc3b5a-cd0c-466e-99ab-8233b3ac089a,ben_lesser_avatar_task,israel,What happened next?,After the horrors of the death train and the l...,2026-09-03 18:23:42.143000+00:00



Automatic opening-pair audit:
Automatic user greetings retained in full audit table: 354
Agent replies to automatic greetings retained in full audit table: 350


## 7. Validate Main-CSV Input Modality and Metadata Matching

Validate typed/voice labels against auxiliary metadata where matching records are available.


In [8]:
merge_summary = (
    messages_labeled["input_method_source"]
    .fillna("missing").value_counts(dropna=False)
    .rename_axis("source").reset_index(name="n_messages")
)
display(merge_summary)

unknown_before_cleaning = (
    ~messages_labeled["input_method"].astype(str).str.lower().isin(["typed", "voice"])
).sum()
print("Messages without a recognized direct voice/typed label:", int(unknown_before_cleaning))

assert unknown_before_cleaning == 0
assert set(messages_labeled["input_method_source"].unique()) == {"main_event_payload"}

display(messages_labeled[[
    "user_id", "stage_id", "user_question_text",
    "input_method", "input_method_source"
]].head(15))

,source,n_messages
0,main_event_payload,2560


Messages without a recognized direct voice/typed label: 0


,user_id,stage_id,user_question_text,input_method,input_method_source
0,001b8917-b3bf-420a-bab6-a683e61fbda8,ben_lesser_avatar_task,"Hello, My name is XXXX XXXX, please tell me ab...",typed,main_event_payload
1,001b8917-b3bf-420a-bab6-a683e61fbda8,ben_lesser_avatar_task,what do you want to tell me about your experience,typed,main_event_payload
2,001b8917-b3bf-420a-bab6-a683e61fbda8,ben_lesser_avatar_task,what is important to you that you want to pass...,typed,main_event_payload
3,001b8917-b3bf-420a-bab6-a683e61fbda8,ben_lesser_avatar_task,what do you want us to remember about your family,typed,main_event_payload
4,001b8917-b3bf-420a-bab6-a683e61fbda8,ben_lesser_avatar_task,Scrub your life before everything started.,voice,main_event_payload
5,001b8917-b3bf-420a-bab6-a683e61fbda8,ben_lesser_avatar_task,describe,typed,main_event_payload
6,001b8917-b3bf-420a-bab6-a683e61fbda8,ben_lesser_avatar_task,What's the first thing that. You remember. Fe...,voice,main_event_payload
7,001b8917-b3bf-420a-bab6-a683e61fbda8,ben_lesser_avatar_task,What happened next?,voice,main_event_payload
8,001b8917-b3bf-420a-bab6-a683e61fbda8,markel_chat_task,"Hello, My name is XXXX XXXX, please tell me ab...",typed,main_event_payload
9,001b8917-b3bf-420a-bab6-a683e61fbda8,markel_chat_task,tell,voice,main_event_payload


## 8. Remove Automatic Openings and Validate the Already-Final Analytic Sample

Remove automatic opening exchanges and confirm the already-final 146-participant analytic sample.


In [9]:
messages_clean_all, messages_removed = hoa.prepare_messages(messages_labeled)

print("Participant messages before automatic-opening removal:", len(messages_labeled))
print("Genuine nonblank messages after automatic-opening removal:", len(messages_clean_all))
print("Automatic or blank messages removed:", len(messages_removed))

removed_summary = pd.DataFrame({
    "automatic_intro": [int(messages_removed["is_auto_intro"].sum())],
    "blank": [int(messages_removed["is_blank"].sum())],
})
display(removed_summary)

real_message_counts = (
    messages_clean_all.loc[messages_clean_all["condition"].isin(["Chat", "Avatar"])]
    .groupby(["user_id", "session_id", "condition"]).size()
    .unstack(fill_value=0).reindex(columns=["Chat", "Avatar"], fill_value=0)
    .reset_index().rename(columns={"Chat": "n_real_chat_messages", "Avatar": "n_real_avatar_messages"})
)
all_participant_sessions = study[["user_id", "session_id"]].drop_duplicates()
participant_interface_eligibility = all_participant_sessions.merge(
    real_message_counts, on=["user_id", "session_id"], how="left", validate="one_to_one"
)
for col in ["n_real_chat_messages", "n_real_avatar_messages"]:
    participant_interface_eligibility[col] = pd.to_numeric(
        participant_interface_eligibility[col], errors="coerce"
    ).fillna(0).astype(int)
participant_interface_eligibility["eligible_both_interfaces"] = (
    participant_interface_eligibility["n_real_chat_messages"].ge(1)
    & participant_interface_eligibility["n_real_avatar_messages"].ge(1)
)
participant_interface_eligibility["exclusion_reason"] = np.where(
    participant_interface_eligibility["eligible_both_interfaces"],
    "retained_in_final_input", "ERROR_incomplete_final_input"
)
excluded_incomplete_interfaces = participant_interface_eligibility.loc[
    ~participant_interface_eligibility["eligible_both_interfaces"]
].copy()

assert excluded_incomplete_interfaces.empty, (
    "The supplied final CSV is not fully curated: at least one participant lacks "
    "a genuine Chat or Avatar message."
)
messages_clean = messages_clean_all.copy()

conditions_per_participant = messages_clean.groupby(["user_id", "session_id"])["condition"].agg(set)
assert conditions_per_participant.map(lambda s: {"Chat", "Avatar"}.issubset(s)).all()

final_sample_n = study["user_id"].nunique()
assert final_sample_n == 146
final_country_counts = (
    study[["user_id", "country"]].drop_duplicates()
    .groupby("country").size().reset_index(name="n_participants")
)

print("\nFINAL ANALYTIC SAMPLE (already materialized in input CSV)")
print("Participants:", final_sample_n)
print("Participants removed by notebook:", 0)
print("\nParticipants by country:")
display(final_country_counts)
print("\nValidation of genuine messages by condition:")
display(messages_clean.groupby("condition").agg(
    n_real_messages=("event_id", "size"), n_participants=("user_id", "nunique")
).reset_index())

assert set(final_country_counts["country"]) == {"israel", "germany", "china"}

Participant messages before automatic-opening removal: 2560
Genuine nonblank messages after automatic-opening removal: 2206
Automatic or blank messages removed: 354


,automatic_intro,blank
0,354,0



FINAL ANALYTIC SAMPLE (already materialized in input CSV)
Participants: 146
Participants removed by notebook: 0

Participants by country:


,country,n_participants
0,china,54
1,germany,43
2,israel,49



Validation of genuine messages by condition:


,condition,n_real_messages,n_participants
0,Avatar,1106,146
1,Chat,1100,146


### Final sample used below

All subsequent analyses use the same retained 146-participant, two-condition sample.


## 9. Inspect Voice and Typed Input Distribution

Summarize observed typed and spoken input use before constructing condition-level metrics.


In [10]:
input_counts = (
    messages_clean.groupby(["country", "condition", "input_method"], dropna=False)
    .size()
    .reset_index(name="n_messages")
)

input_counts["percent_within_country_condition"] = (
    input_counts["n_messages"]
    / input_counts.groupby(["country", "condition"])["n_messages"].transform("sum")
    * 100
)

display(input_counts)

input_patterns = (
    messages_clean.groupby(
        ["country", "condition", "user_id", "session_id", "stage_id"],
        dropna=False
    )["input_method"]
    .agg(
        lambda s: (
            "mixed"
            if set(s.dropna().astype(str).str.lower()) >= {"voice", "typed"}
            else "voice_only"
            if set(s.dropna().astype(str).str.lower()) == {"voice"}
            else "typed_only"
            if set(s.dropna().astype(str).str.lower()) == {"typed"}
            else "unknown"
        )
    )
    .reset_index(name="usage_pattern")
)

display(
    input_patterns.groupby(["country", "condition", "usage_pattern"])
    .size()
    .reset_index(name="n_interactions")
)


,country,condition,input_method,n_messages,percent_within_country_condition
0,china,Avatar,typed,251,77.950311
1,china,Avatar,voice,71,22.049689
2,china,Chat,typed,273,94.137931
3,china,Chat,voice,17,5.862069
4,germany,Avatar,typed,192,52.173913
5,germany,Avatar,voice,176,47.826087
6,germany,Chat,typed,358,100.000000
7,israel,Avatar,typed,57,13.701923
8,israel,Avatar,voice,359,86.298077
9,israel,Chat,typed,403,89.159292


,country,condition,usage_pattern,n_interactions
0,china,Avatar,mixed,5
1,china,Avatar,typed_only,40
2,china,Avatar,voice_only,9
3,china,Chat,mixed,8
4,china,Chat,typed_only,44
5,china,Chat,voice_only,2
6,germany,Avatar,mixed,4
7,germany,Avatar,typed_only,21
8,germany,Avatar,voice_only,18
9,germany,Chat,typed_only,43


## 10. Compute Interaction-Level Metrics

Aggregate event-level logs into participant-by-condition behavioral measures.


In [11]:
interaction_metrics = hoa.build_interaction_metrics(messages_clean)

print("Number of interaction units:", len(interaction_metrics))
print("Number of participants:", interaction_metrics["user_id"].nunique())

display(interaction_metrics.head())

display(
    interaction_metrics.groupby(["country", "condition"])
    .agg(
        n_interactions=("user_id", "size"),
        n_participants=("user_id", "nunique"),
        mean_messages=("n_messages", "mean"),
        mean_words=("total_words", "mean"),
        mean_voice_rate=("voice_message_rate", "mean"),
        mean_duration_minutes=("interaction_duration_minutes", "mean"),
        mean_agent_words=("mean_agent_words_per_response", "mean"),
    )
    .reset_index()
)


Number of interaction units: 292
Number of participants: 146


,user_id,session_id,stage_id,character,condition,country,n_messages,n_voice_messages,n_typed_messages,n_unknown_input_messages,...,agent_response_coverage,total_agent_words,mean_agent_words_per_response,median_agent_response_latency_seconds,mean_agent_response_latency_seconds,median_avg_ping_ms,mean_avg_ping_ms,user_to_agent_word_ratio,first_real_message_time,last_real_message_time
0,001b8917-b3bf-420a-bab6-a683e61fbda8,45fc3b5a-cd0c-466e-99ab-8233b3ac089a,ben_lesser_avatar_task,Ben Lesser,Avatar,israel,7,3,4,0,...,1.0,367.0,52.428571,1.9700,1.979714,138.0,139.428571,0.144414,2026-09-03 18:20:03.817000+00:00,2026-09-03 18:23:42.143000+00:00
1,001b8917-b3bf-420a-bab6-a683e61fbda8,45fc3b5a-cd0c-466e-99ab-8233b3ac089a,markel_chat_task,Markel,Chat,israel,8,1,7,0,...,1.0,313.0,39.125000,1.6585,1.706250,151.0,138.875000,0.156550,2026-09-03 18:26:29.093000+00:00,2026-09-03 18:29:28.989000+00:00
2,005ff203-092d-4f07-9960-c28ad30d7289,44f2472e-5bfe-40ee-afad-6352f037bdcf,ben_lesser_chat_task,Ben Lesser,Chat,israel,6,4,2,0,...,1.0,195.0,32.500000,1.5000,1.460333,106.0,118.166667,0.282051,2026-04-19 19:04:06.914000+00:00,2026-04-19 19:08:07.208000+00:00
3,005ff203-092d-4f07-9960-c28ad30d7289,44f2472e-5bfe-40ee-afad-6352f037bdcf,markel_avatar_task,Markel,Avatar,israel,5,5,0,0,...,1.0,159.0,31.800000,1.2650,1.304600,64.0,64.400000,0.440252,2026-04-19 18:51:53.398000+00:00,2026-04-19 18:55:49.678000+00:00
4,045f1c9f-9770-4b41-b08b-101d26a042af,c0449a2a-b41a-4d83-8e5d-0522397c543b,ben_lesser_chat_task,Ben Lesser,Chat,china,4,0,4,0,...,1.0,211.0,52.750000,2.1595,2.191750,167.0,193.250000,0.132701,2026-08-28 03:48:29.727000+00:00,2026-08-28 03:50:28.024000+00:00


,country,condition,n_interactions,n_participants,mean_messages,mean_words,mean_voice_rate,mean_duration_minutes,mean_agent_words
0,china,Avatar,54,54,5.962963,62.907407,0.210185,3.707525,42.666387
1,china,Chat,54,54,5.370370,50.740741,0.070032,3.243058,43.885346
2,germany,Avatar,43,43,8.558140,87.209302,0.467499,4.546935,44.473987
3,germany,Chat,43,43,8.325581,78.558140,0.000000,4.054471,42.933685
4,israel,Avatar,49,49,8.489796,76.183673,0.863338,4.178669,32.056129
5,israel,Chat,49,49,9.224490,82.632653,0.131468,4.770427,31.402056


## 11. Extract Questionnaire Scales, Importance Change, Demographics, and Knowledge

Construct the questionnaire, importance-change, demographic, and knowledge variables used in the manuscript analyses.

In [12]:
assignments = hoa.extract_assignments(study)
demographics = hoa.extract_demographics(study).drop(
    columns=["country"], errors="ignore"
)
knowledge_items, knowledge_scores = hoa.extract_knowledge_items(study)

# Exact multi-item post-interaction scales.
scale_items, questionnaire_scales = hoa.extract_character_scales(study)

# Initial importance, post-character importance, and change from baseline.
importance_items, importance_scores = hoa.extract_importance_ratings(study)

# Reliability table for the reported multi-item questionnaire scales.
reliability = hoa.questionnaire_reliability(scale_items)

print("Assignment rows:", len(assignments))
print("Demographic rows:", len(demographics))
print("Knowledge-item rows:", len(knowledge_items))
print("Participant-by-character knowledge scores:", len(knowledge_scores))
print("Questionnaire-item rows:", len(scale_items))
print("Importance post-character rows:", len(importance_scores))

print("\nQuestionnaire reliability:")
display(reliability)

print("\nImportance scoring preview:")
display(importance_scores.head())

display(questionnaire_scales.head())

Assignment rows: 146
Demographic rows: 146
Knowledge-item rows: 1460
Participant-by-character knowledge scores: 292
Questionnaire-item rows: 1460
Importance post-character rows: 292

Questionnaire reliability:


,scale,character,n_items,n_complete,cronbach_alpha,spearman_brown
0,character_connection,Ben Lesser,2,146,0.864454,0.864474
1,character_connection,Markel,2,146,0.872395,0.872831
2,satisfaction,Ben Lesser,3,146,0.922266,NaN
3,satisfaction,Markel,3,146,0.923948,NaN



Importance scoring preview:


,user_id,session_id,character,initial_importance,post_importance,importance_change
0,501709ee-e9dd-4dd0-a16d-127521471879,e46ca164-d910-4dd4-a6f9-379f49658f77,Markel,5.0,5.0,0.0
1,501709ee-e9dd-4dd0-a16d-127521471879,e46ca164-d910-4dd4-a6f9-379f49658f77,Ben Lesser,5.0,5.0,0.0
2,80c75c1f-14b6-483d-84b2-d734094721ca,a0fcfb8d-ace0-48fc-a47d-cb400af9168b,Ben Lesser,4.0,4.0,0.0
3,80c75c1f-14b6-483d-84b2-d734094721ca,a0fcfb8d-ace0-48fc-a47d-cb400af9168b,Markel,4.0,4.0,0.0
4,40108167-a891-44f1-8522-b30de68dc2f8,8681799e-f9a6-42e4-9016-e9aeb54a58e4,Ben Lesser,5.0,5.0,0.0


,user_id,session_id,character,character_connection,satisfaction
0,001b8917-b3bf-420a-bab6-a683e61fbda8,45fc3b5a-cd0c-466e-99ab-8233b3ac089a,Ben Lesser,3.0,2.0
1,001b8917-b3bf-420a-bab6-a683e61fbda8,45fc3b5a-cd0c-466e-99ab-8233b3ac089a,Markel,2.0,3.0
2,005ff203-092d-4f07-9960-c28ad30d7289,44f2472e-5bfe-40ee-afad-6352f037bdcf,Ben Lesser,4.0,4.0
3,005ff203-092d-4f07-9960-c28ad30d7289,44f2472e-5bfe-40ee-afad-6352f037bdcf,Markel,3.5,4.0
4,045f1c9f-9770-4b41-b08b-101d26a042af,c0449a2a-b41a-4d83-8e5d-0522397c543b,Ben Lesser,4.0,3.0


### Participant age descriptives

Summarize age in the retained analytic sample.


In [13]:
# Participant Age Descriptives

print("Available demographic columns:")
print(demographics.columns.tolist())

age_candidates = [
    col for col in demographics.columns
    if "age_numeric" in col.lower()
]

if len(age_candidates) != 1:
    raise ValueError(
        f"Expected exactly one age column, found: {age_candidates}"
    )

AGE_COL = age_candidates[0]

age_values = pd.to_numeric(
    demographics[AGE_COL],
    errors="coerce"
).dropna()

age_summary = pd.DataFrame({
    "n_participants_with_age": [len(age_values)],
    "mean_age": [age_values.mean()],
    "sd_age": [age_values.std(ddof=1)],
    "min_age": [age_values.min()],
    "max_age": [age_values.max()],
})

age_summary = age_summary.round(2)

print("\nAge descriptives - final analytic sample:")
display(age_summary)

Available demographic columns:
['user_id', 'session_id', 'age', 'ai_experience_years', 'educational_background', 'explanation_type_preference', 'gender', 'interface_detail_preference', 'participation_consent', 'professional_background', 'age_numeric', 'ai_experience_years_numeric']

Age descriptives - final analytic sample:


,n_participants_with_age,mean_age,sd_age,min_age,max_age
0,146,30.11,7.72,21.0,65.0


### Participant gender descriptives

Summarize gender composition in the retained analytic sample.


In [14]:
# Gender distribution
gender_col_candidates = [
    col for col in demographics.columns
    if "gender" in col.lower()
]

print("\nPossible gender columns:")
print(gender_col_candidates)

if gender_col_candidates:
    GENDER_COL = gender_col_candidates[0]

    gender_distribution = (
        demographics[GENDER_COL]
        .fillna("Missing")
        .value_counts(dropna=False)
        .rename_axis("gender")
        .reset_index(name="n")
    )

    gender_distribution["percent"] = (
        100 * gender_distribution["n"] / len(demographics)
    ).round(1)

    print("\nGender distribution:")
    display(gender_distribution)



Possible gender columns:
['gender']

Gender distribution:


,gender,n,percent
0,male,111,76.0
1,female,33,22.6
2,other,2,1.4


## 12. Build the Corrected Merged Analysis Dataset

Merge questionnaire, knowledge, demographic, and interaction-level measures into the canonical analysis dataset.


In [15]:
analysis = hoa.attach_design_information(
    interaction_metrics,
    assignments,
    demographics,
)

analysis = analysis.merge(
    knowledge_scores,
    on=["user_id", "session_id", "character"],
    how="left",
)

analysis = analysis.merge(
    questionnaire_scales,
    on=["user_id", "session_id", "character"],
    how="left",
)

analysis = analysis.merge(
    importance_scores,
    on=["user_id", "session_id", "character"],
    how="left",
)


analysis = hoa.add_normalized_gender(analysis)

analysis_key = ["user_id", "session_id", "character", "condition"]
duplicate_keys = analysis.duplicated(analysis_key).sum()

print("Merged analysis dataset shape:", analysis.shape)
print("Duplicate rows by analysis key:", int(duplicate_keys))
print("Participants by country:")
display(
    analysis.groupby("country")["user_id"]
    .nunique()
    .reset_index(name="n_participants")
)

print("\nExact outcome completeness:")
display(
    analysis[[
        "knowledge_accuracy",
        "satisfaction",
        "character_connection",
        "initial_importance",
        "post_importance",
        "importance_change",
    ]].notna().sum().rename("n_non_missing").reset_index()
)

assert duplicate_keys == 0, "Duplicate interaction rows were found."
display(analysis.head())


Merged analysis dataset shape: (292, 81)
Duplicate rows by analysis key: 0
Participants by country:


,country,n_participants
0,china,54
1,germany,43
2,israel,49



Exact outcome completeness:


,index,n_non_missing
0,knowledge_accuracy,292
1,satisfaction,292
2,character_connection,292
3,initial_importance,292
4,post_importance,292
5,importance_change,292


,user_id,session_id,stage_id,character,condition,country,n_messages,n_voice_messages,n_typed_messages,n_unknown_input_messages,...,knowledge_correct,knowledge_n_items,knowledge_accuracy,knowledge_completed_at,character_connection,satisfaction,initial_importance,post_importance,importance_change,gender_normalized
0,001b8917-b3bf-420a-bab6-a683e61fbda8,45fc3b5a-cd0c-466e-99ab-8233b3ac089a,ben_lesser_avatar_task,Ben Lesser,Avatar,israel,7,3,4,0,...,4,5,0.8,2026-09-03 18:24:58.903994+00:00,3.0,2.0,5.0,4.0,-1.0,male
1,001b8917-b3bf-420a-bab6-a683e61fbda8,45fc3b5a-cd0c-466e-99ab-8233b3ac089a,markel_chat_task,Markel,Chat,israel,8,1,7,0,...,2,5,0.4,2026-09-03 18:31:09.235773+00:00,2.0,3.0,5.0,5.0,0.0,male
2,005ff203-092d-4f07-9960-c28ad30d7289,44f2472e-5bfe-40ee-afad-6352f037bdcf,ben_lesser_chat_task,Ben Lesser,Chat,israel,6,4,2,0,...,3,5,0.6,2026-04-19 19:09:43.799828+00:00,4.0,4.0,5.0,5.0,0.0,female
3,005ff203-092d-4f07-9960-c28ad30d7289,44f2472e-5bfe-40ee-afad-6352f037bdcf,markel_avatar_task,Markel,Avatar,israel,5,5,0,0,...,4,5,0.8,2026-04-19 19:00:03.971818+00:00,3.5,4.0,5.0,5.0,0.0,female
4,045f1c9f-9770-4b41-b08b-101d26a042af,c0449a2a-b41a-4d83-8e5d-0522397c543b,ben_lesser_chat_task,Ben Lesser,Chat,china,4,0,4,0,...,3,5,0.6,2026-08-28 17:20:59.677097+00:00,4.0,3.0,4.0,4.0,0.0,male


### Baseline importance and ceiling-effect audit

Quantify baseline importance and ceiling exposure relevant to interpreting change scores.


In [16]:
# Importance baseline / ceiling-effect audit

from scipy import stats
from statsmodels.stats.multitest import multipletests
from itertools import combinations

# keep ONE baseline observation per participant.
importance_baseline = (
    analysis[
        ["user_id", "session_id", "country", "initial_importance"]
    ]
    .dropna(subset=["initial_importance", "country"])
    .drop_duplicates(["user_id", "session_id"])
    .copy()
)

importance_baseline["at_ceiling"] = (
    importance_baseline["initial_importance"] == 5
)

# Overall baseline
print("OVERALL BASELINE IMPORTANCE")
print(
    f"N={len(importance_baseline)}, "
    f"M={importance_baseline['initial_importance'].mean():.3f}, "
    f"SD={importance_baseline['initial_importance'].std(ddof=1):.3f}, "
    f"Median={importance_baseline['initial_importance'].median():.1f}, "
    f"% at maximum (5/5)="
    f"{100 * importance_baseline['at_ceiling'].mean():.1f}%"
)

# Baseline by country
baseline_country = (
    importance_baseline
    .groupby("country")
    .agg(
        n=("initial_importance", "size"),
        mean=("initial_importance", "mean"),
        sd=("initial_importance", "std"),
        median=("initial_importance", "median"),
        n_at_ceiling=("at_ceiling", "sum"),
        prop_at_ceiling=("at_ceiling", "mean"),
    )
    .reset_index()
)

baseline_country["percent_at_ceiling"] = (
    100 * baseline_country["prop_at_ceiling"]
)

print("\nBASELINE IMPORTANCE BY COUNTRY")
display(
    baseline_country[
        [
            "country", "n", "mean", "sd", "median",
            "n_at_ceiling", "percent_at_ceiling"
        ]
    ]
)

# Country difference in baseline importance
# Appropriate sensitivity test for ordinal 1–5 ratings
country_groups = [
    g["initial_importance"].to_numpy()
    for _, g in importance_baseline.groupby("country")
]

kw_stat, kw_p = stats.kruskal(*country_groups)

print(
    f"\nKruskal-Wallis country comparison: "
    f"H={kw_stat:.3f}, p={kw_p:.4f}"
)

# Do countries differ in the proportion already at ceiling?
ceiling_table = pd.crosstab(
    importance_baseline["country"],
    importance_baseline["at_ceiling"]
)

chi2, chi_p, dof, expected = stats.chi2_contingency(ceiling_table)

print(
    f"Country × ceiling-status chi-square: "
    f"chi2={chi2:.3f}, df={dof}, p={chi_p:.4f}"
)

# Pairwise country comparisons, Holm-corrected
pairwise = []

for a, b in combinations(sorted(importance_baseline["country"].unique()), 2):
    x = importance_baseline.loc[
        importance_baseline["country"] == a,
        "initial_importance"
    ]
    y = importance_baseline.loc[
        importance_baseline["country"] == b,
        "initial_importance"
    ]

    u, p = stats.mannwhitneyu(x, y, alternative="two-sided")

    pairwise.append({
        "country_a": a,
        "country_b": b,
        "mean_a": x.mean(),
        "mean_b": y.mean(),
        "U": u,
        "p": p,
    })

pairwise = pd.DataFrame(pairwise)

pairwise["p_holm"] = multipletests(
    pairwise["p"],
    method="holm"
)[1]

print("\nPAIRWISE BASELINE COUNTRY COMPARISONS")
display(pairwise)

# Post-interaction raw importance values by country and condition
# (not change scores)
post_country_condition = (
    analysis
    .groupby(["country", "condition"])
    .agg(
        n=("post_importance", "count"),
        mean_post=("post_importance", "mean"),
        sd_post=("post_importance", "std"),
        median_post=("post_importance", "median"),
        percent_post_at_5=(
            "post_importance",
            lambda x: 100 * (x == 5).mean()
        ),
    )
    .reset_index()
)

print("\nPOST-INTERACTION IMPORTANCE VALUES BY COUNTRY × CONDITION")
display(post_country_condition)


OVERALL BASELINE IMPORTANCE
N=146, M=4.418, SD=0.759, Median=5.0, % at maximum (5/5)=56.8%

BASELINE IMPORTANCE BY COUNTRY


,country,n,mean,sd,median,n_at_ceiling,percent_at_ceiling
0,china,54,4.166667,0.818420,4.0,21,38.888889
1,germany,43,4.465116,0.701998,5.0,25,58.139535
2,israel,49,4.653061,0.663043,5.0,37,75.510204



Kruskal-Wallis country comparison: H=12.395, p=0.0020
Country × ceiling-status chi-square: chi2=14.086, df=2, p=0.0009

PAIRWISE BASELINE COUNTRY COMPARISONS


,country_a,country_b,mean_a,mean_b,U,p,p_holm
0,china,germany,4.166667,4.465116,925.0,0.062035,0.124070
1,china,israel,4.166667,4.653061,856.0,0.000549,0.001648
2,germany,israel,4.465116,4.653061,885.5,0.112582,0.124070



POST-INTERACTION IMPORTANCE VALUES BY COUNTRY × CONDITION


,country,condition,n,mean_post,sd_post,median_post,percent_post_at_5
0,china,Avatar,54,4.185185,0.728764,4.0,35.185185
1,china,Chat,54,4.129630,0.753515,4.0,33.333333
2,germany,Avatar,43,4.534884,0.667220,5.0,62.790698
3,germany,Chat,43,4.558140,0.628768,5.0,62.790698
4,israel,Avatar,49,4.673469,0.554818,5.0,71.428571
5,israel,Chat,49,4.591837,0.674486,5.0,69.387755


# Part I - Manuscript Analysis Plan

Define the manuscript-facing outcome families, multiplicity corrections, and the mixed primary-analysis design. Character connection, satisfaction, and knowledge remain within-participant Avatar-versus-Chat outcomes; perceived-importance change is analyzed only after the first interaction to avoid carryover from the preceding condition.


In [17]:
from itertools import combinations
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

# Three primary outcomes remain valid within-participant comparisons.
PAIRED_PRIMARY_OUTCOMES = [
    "character_connection",
    "satisfaction",
    "knowledge_accuracy",
]

# Importance is a global attitude measured once at baseline and after each
# interaction. To avoid contamination/carryover from interaction 1 into
# interaction 2, the confirmatory importance analysis uses ONLY the first
# interaction for each participant.
FIRST_INTERACTION_PRIMARY_OUTCOME = "importance_change"
PRIMARY_OUTCOMES = PAIRED_PRIMARY_OUTCOMES + [FIRST_INTERACTION_PRIMARY_OUTCOME]

importance_first = (
    analysis.loc[
        analysis["condition_order"].eq("first"),
        [
            "user_id", "session_id", "country", "character", "condition",
            "condition_order", "condition_position", "gender_normalized",
            "initial_importance", "post_importance",
            "importance_change",
        ],
    ]
    .dropna(subset=["condition", "initial_importance", "post_importance", "importance_change"])
    .copy()
)

# One and only one first interaction per retained participant-session.
_first_counts = importance_first.groupby(["user_id", "session_id"]).size()
assert len(importance_first) == analysis[["user_id", "session_id"]].drop_duplicates().shape[0]
assert (_first_counts == 1).all(), "Expected exactly one first interaction per participant-session."
assert set(importance_first["condition"].unique()) == {"Avatar", "Chat"}

FIRST_CONDITION_COUNTS = importance_first["condition"].value_counts().to_dict()
print("First-interaction importance sample by condition:", FIRST_CONDITION_COUNTS)


def _holm_series(values):
    values = pd.Series(values, dtype=float)
    out = pd.Series(np.nan, index=values.index, dtype=float)
    mask = values.notna()
    if mask.any():
        out.loc[mask] = multipletests(values.loc[mask], method="holm")[1]
    return out


def _welch_difference_ci(a, b, alpha=0.05):
    """Welch CI for mean(a) - mean(b)."""
    a = pd.to_numeric(pd.Series(a), errors="coerce").dropna().astype(float)
    b = pd.to_numeric(pd.Series(b), errors="coerce").dropna().astype(float)
    va, vb = a.var(ddof=1), b.var(ddof=1)
    se2 = va / len(a) + vb / len(b)
    denom = ((va / len(a)) ** 2 / (len(a) - 1)) + ((vb / len(b)) ** 2 / (len(b) - 1))
    df = (se2 ** 2) / denom if se2 > 0 and denom > 0 else np.nan
    diff = float(a.mean() - b.mean())
    if np.isfinite(df) and se2 > 0:
        tcrit = stats.t.ppf(1 - alpha / 2, df)
        half = float(tcrit * np.sqrt(se2))
        return diff, diff - half, diff + half, float(df)
    return diff, np.nan, np.nan, float(df) if np.isfinite(df) else np.nan


def _cohens_d_independent(a, b):
    a = pd.to_numeric(pd.Series(a), errors="coerce").dropna().astype(float)
    b = pd.to_numeric(pd.Series(b), errors="coerce").dropna().astype(float)
    if len(a) < 2 or len(b) < 2:
        return np.nan
    va, vb = a.var(ddof=1), b.var(ddof=1)
    pooled_var = ((len(a) - 1) * va + (len(b) - 1) * vb) / (len(a) + len(b) - 2)
    if not np.isfinite(pooled_var) or pooled_var <= 0:
        return np.nan
    return float((a.mean() - b.mean()) / np.sqrt(pooled_var))


def first_interaction_importance_test(data, min_group_n=5):
    """Between-participant comparison of baseline-to-post change after interaction 1 only."""
    frame = data.loc[data["condition_order"].eq("first")].copy()
    frame["importance_change"] = pd.to_numeric(frame["importance_change"], errors="coerce")
    frame = frame.dropna(subset=["importance_change", "condition"])

    avatar = frame.loc[frame["condition"].eq("Avatar"), "importance_change"]
    chat = frame.loc[frame["condition"].eq("Chat"), "importance_change"]
    if len(avatar) < min_group_n or len(chat) < min_group_n:
        return pd.DataFrame()

    welch = stats.ttest_ind(avatar, chat, equal_var=False, nan_policy="omit")
    diff, ci_low, ci_high, welch_df = _welch_difference_ci(avatar, chat)
    try:
        u_stat, u_p = stats.mannwhitneyu(avatar, chat, alternative="two-sided")
    except Exception:
        u_stat, u_p = np.nan, np.nan

    return pd.DataFrame([{
        "outcome": "importance_change",
        "analysis_design": "between_participants_first_interaction_only",
        "n_pairs": np.nan,
        "n_avatar": int(len(avatar)),
        "n_chat": int(len(chat)),
        "avatar_mean": float(avatar.mean()),
        "avatar_sd": float(avatar.std(ddof=1)),
        "chat_mean": float(chat.mean()),
        "chat_sd": float(chat.std(ddof=1)),
        "mean_difference_avatar_minus_chat": diff,
        "difference_ci95_low": ci_low,
        "difference_ci95_high": ci_high,
        "paired_t": np.nan,
        "welch_t": float(welch.statistic),
        "welch_df": welch_df,
        "p_value": float(welch.pvalue),
        "cohens_dz": np.nan,
        "cohens_d": _cohens_d_independent(avatar, chat),
        "wilcoxon_w": np.nan,
        "wilcoxon_p": np.nan,
        "rank_biserial": np.nan,
        "mann_whitney_u": float(u_stat) if np.isfinite(u_stat) else np.nan,
        "mann_whitney_p": float(u_p) if np.isfinite(u_p) else np.nan,
        "rank_test": "Mann-Whitney U",
        "rank_test_p": float(u_p) if np.isfinite(u_p) else np.nan,
        "difference_shapiro_w": np.nan,
        "difference_shapiro_p": np.nan,
    }])


def first_interaction_baseline_balance(data, min_group_n=5):
    """Diagnostic comparison of the single pre-interaction baseline by first assigned condition."""
    frame = data.loc[data["condition_order"].eq("first")].copy()
    avatar = pd.to_numeric(
        frame.loc[frame["condition"].eq("Avatar"), "initial_importance"], errors="coerce"
    ).dropna()
    chat = pd.to_numeric(
        frame.loc[frame["condition"].eq("Chat"), "initial_importance"], errors="coerce"
    ).dropna()
    if len(avatar) < min_group_n or len(chat) < min_group_n:
        return pd.DataFrame()
    welch = stats.ttest_ind(avatar, chat, equal_var=False)
    diff, ci_low, ci_high, df = _welch_difference_ci(avatar, chat)
    u, p_u = stats.mannwhitneyu(avatar, chat, alternative="two-sided")
    return pd.DataFrame([{
        "comparison": "baseline_by_first_assigned_condition",
        "n_avatar_first": len(avatar),
        "n_chat_first": len(chat),
        "avatar_first_baseline_mean": avatar.mean(),
        "chat_first_baseline_mean": chat.mean(),
        "mean_difference": diff,
        "ci95_low": ci_low,
        "ci95_high": ci_high,
        "welch_t": welch.statistic,
        "welch_df": df,
        "p_value": welch.pvalue,
        "mann_whitney_u": u,
        "mann_whitney_p": p_u,
    }])

# Secondary behavioral multiplicity family used in the manuscript.
SECONDARY_BEHAVIOR_OUTCOMES = [
    "n_messages",
    "total_words",
    "mean_words_per_message",
    "unique_words",
    "conversation_type_token_ratio",
    "n_objective_question_forms",
    "question_form_rate",
    "interaction_duration_minutes",
    "messages_per_minute",
    "voice_message_rate",
    "n_voice_messages",
    "n_typed_messages",
    "n_input_switches",
    "mean_agent_words_per_response",
    "mean_agent_response_latency_seconds",
    "agent_response_coverage",
    "user_to_agent_word_ratio",
]

EXPLORATORY_BEHAVIOR_OUTCOMES = []

COUNTRY_ORDER = [
    c for c in ["israel", "germany", "china"]
    if c in set(analysis["country"].dropna().astype(str))
]
assert len(COUNTRY_ORDER) == 3, f"Expected three groups, got {COUNTRY_ORDER}"

print("Paired primary outcomes:", PAIRED_PRIMARY_OUTCOMES)
print("First-interaction-only primary outcome:", FIRST_INTERACTION_PRIMARY_OUTCOME)
print("Primary Holm family:", PRIMARY_OUTCOMES)
print("Secondary behavioral family:", SECONDARY_BEHAVIOR_OUTCOMES)
print("Country order:", COUNTRY_ORDER)


First-interaction importance sample by condition: {'Avatar': 73, 'Chat': 73}
Paired primary outcomes: ['character_connection', 'satisfaction', 'knowledge_accuracy']
First-interaction-only primary outcome: importance_change
Primary Holm family: ['character_connection', 'satisfaction', 'knowledge_accuracy', 'importance_change']
Secondary behavioral family: ['n_messages', 'total_words', 'mean_words_per_message', 'unique_words', 'conversation_type_token_ratio', 'n_objective_question_forms', 'question_form_rate', 'interaction_duration_minutes', 'messages_per_minute', 'voice_message_rate', 'n_voice_messages', 'n_typed_messages', 'n_input_switches', 'mean_agent_words_per_response', 'mean_agent_response_latency_seconds', 'agent_response_coverage', 'user_to_agent_word_ratio']
Country order: ['israel', 'germany', 'china']


# Part II - Whole-Sample Primary Effects

Estimate the three valid within-participant Avatar-versus-Chat primary effects and the first-interaction-only between-participant importance effect, then apply Holm correction across the four-outcome primary family.


In [18]:
# Three outcomes use the original within-participant design.
paired_primary = hoa.paired_condition_tests(
    analysis, PAIRED_PRIMARY_OUTCOMES, MIN_GROUP_N
).copy()
if not paired_primary.empty:
    paired_primary["analysis_design"] = "within_participant_paired"
    paired_primary["n_avatar"] = paired_primary["n_pairs"]
    paired_primary["n_chat"] = paired_primary["n_pairs"]
    paired_primary["welch_t"] = np.nan
    paired_primary["welch_df"] = np.nan
    paired_primary["cohens_d"] = np.nan
    paired_primary["mann_whitney_u"] = np.nan
    paired_primary["mann_whitney_p"] = np.nan
    paired_primary["rank_test"] = "Wilcoxon signed-rank"
    paired_primary["rank_test_p"] = paired_primary["wilcoxon_p"]

# Importance uses only the first interaction, so Avatar-first and Chat-first
# are independent groups (73 participants per group in the final sample).
importance_first_primary = first_interaction_importance_test(
    analysis, MIN_GROUP_N
)

whole_sample_primary = pd.concat(
    [paired_primary, importance_first_primary],
    ignore_index=True,
    sort=False,
)

# Recompute multiplicity correction jointly across all four primary outcomes.
whole_sample_primary["p_holm"] = _holm_series(whole_sample_primary["p_value"])
whole_sample_primary["rank_test_p_holm"] = _holm_series(
    whole_sample_primary["rank_test_p"]
)

# Preserve test-specific corrected columns without mislabeling the importance rank test.
whole_sample_primary["wilcoxon_p_holm"] = np.where(
    whole_sample_primary["analysis_design"].eq("within_participant_paired"),
    whole_sample_primary["rank_test_p_holm"],
    np.nan,
)
whole_sample_primary["mann_whitney_p_holm"] = np.where(
    whole_sample_primary["analysis_design"].eq("between_participants_first_interaction_only"),
    whole_sample_primary["rank_test_p_holm"],
    np.nan,
)

importance_first_interaction_baseline_balance = first_interaction_baseline_balance(
    analysis, MIN_GROUP_N
)

whole_sample_secondary = hoa.paired_condition_tests(
    analysis, SECONDARY_BEHAVIOR_OUTCOMES, MIN_GROUP_N
)

print("PRIMARY OUTCOMES - FULL SAMPLE")
display(
    whole_sample_primary[[
        "outcome", "analysis_design", "n_pairs", "n_avatar", "n_chat",
        "avatar_mean", "chat_mean", "mean_difference_avatar_minus_chat",
        "difference_ci95_low", "difference_ci95_high",
        "paired_t", "welch_t", "welch_df", "p_value", "p_holm",
        "cohens_dz", "cohens_d", "rank_test", "rank_test_p", "rank_test_p_holm",
    ]].sort_values("p_holm", na_position="last")
)

print("FIRST-INTERACTION BASELINE BALANCE DIAGNOSTIC")
display(importance_first_interaction_baseline_balance)

print("SECONDARY BEHAVIORAL OUTCOMES - FULL SAMPLE")
display(
    whole_sample_secondary[[
        "outcome", "n_pairs", "avatar_mean", "chat_mean",
        "mean_difference_avatar_minus_chat",
        "difference_ci95_low", "difference_ci95_high",
        "p_value", "p_holm", "cohens_dz", "wilcoxon_p_holm",
    ]].sort_values("p_holm", na_position="last")
)


PRIMARY OUTCOMES - FULL SAMPLE


,outcome,analysis_design,n_pairs,n_avatar,n_chat,avatar_mean,chat_mean,mean_difference_avatar_minus_chat,difference_ci95_low,difference_ci95_high,paired_t,welch_t,welch_df,p_value,p_holm,cohens_dz,cohens_d,rank_test,rank_test_p,rank_test_p_holm
0,character_connection,within_participant_paired,146.0,146,146,3.630137,3.308219,0.321918,0.196775,0.447060,5.084271,NaN,NaN,0.000001,0.000004,0.420777,NaN,Wilcoxon signed-rank,0.000001,0.000006
1,satisfaction,within_participant_paired,146.0,146,146,3.732877,3.372146,0.360731,0.212684,0.508777,4.815851,NaN,NaN,0.000004,0.000011,0.398563,NaN,Wilcoxon signed-rank,0.000006,0.000017
2,knowledge_accuracy,within_participant_paired,146.0,146,146,0.638356,0.617808,0.020548,-0.025354,0.066450,0.884763,NaN,NaN,0.377749,0.755498,0.073224,NaN,Wilcoxon signed-rank,0.393409,0.786817
3,importance_change,between_participants_first_interaction_only,NaN,73,73,0.068493,0.027397,0.041096,-0.119312,0.201504,NaN,0.50649,140.714165,0.613305,0.755498,NaN,0.083835,Mann-Whitney U,0.689514,0.786817


FIRST-INTERACTION BASELINE BALANCE DIAGNOSTIC


,comparison,n_avatar_first,n_chat_first,avatar_first_baseline_mean,chat_first_baseline_mean,mean_difference,ci95_low,ci95_high,welch_t,welch_df,p_value,mann_whitney_u,mann_whitney_p
0,baseline_by_first_assigned_condition,73,73,4.465753,4.369863,0.09589,-0.152796,0.344577,0.762252,141.591078,0.447178,2799.5,0.553391


SECONDARY BEHAVIORAL OUTCOMES - FULL SAMPLE


,outcome,n_pairs,avatar_mean,chat_mean,mean_difference_avatar_minus_chat,difference_ci95_low,difference_ci95_high,p_value,p_holm,cohens_dz,wilcoxon_p_holm
9,voice_message_rate,146,0.505178,0.070025,0.435153,0.357092,0.513214,6.862300e-21,1.166591e-19,0.911844,7.023672e-14
10,n_voice_messages,146,4.150685,0.452055,3.698630,3.004285,4.392976,1.316772e-19,2.106835e-18,0.871318,5.105268e-13
11,n_typed_messages,146,3.424658,7.082192,-3.657534,-4.610437,-2.704631,3.647403e-12,5.471104e-11,-0.627843,3.351520e-10
0,n_messages,146,7.575342,7.534247,0.041096,-0.649654,0.731846,9.065564e-01,1.000000e+00,0.009732,1.000000e+00
14,mean_agent_response_latency_seconds,146,2.038168,2.423087,-0.384918,-1.028699,0.258863,2.392473e-01,1.000000e+00,-0.097801,1.000000e+00
13,mean_agent_words_per_response,146,39.637785,39.415465,0.222320,-1.118744,1.563385,7.436447e-01,1.000000e+00,0.027117,1.000000e+00
12,n_input_switches,146,0.130137,0.150685,-0.020548,-0.123049,0.081953,6.925319e-01,1.000000e+00,-0.032791,1.000000e+00
15,agent_response_coverage,146,0.999315,0.998034,0.001281,-0.002405,0.004967,4.932349e-01,1.000000e+00,0.056849,1.000000e+00
8,messages_per_minute,145,1.959715,1.995411,-0.035696,-0.224315,0.152922,7.089050e-01,1.000000e+00,-0.031065,1.000000e+00
6,question_form_rate,146,0.792390,0.787721,0.004668,-0.035545,0.044881,8.188436e-01,1.000000e+00,0.018989,1.000000e+00


# Part III - Three-Group Country Moderation

Test whether participant-level Avatar-minus-Chat effects differ across the three research cohorts.


In [19]:
def _holm(values):
    return _holm_series(values)


def _joint_wald(model, parameter_names):
    names = list(model.params.index)
    selected = [n for n in parameter_names if n in names]
    if not selected:
        return np.nan, 0, np.nan
    R = np.zeros((len(selected), len(names)))
    for row, name in enumerate(selected):
        R[row, names.index(name)] = 1.0
    test = model.wald_test(R, scalar=True)
    return float(np.asarray(test.statistic).squeeze()), len(selected), float(np.asarray(test.pvalue).squeeze())


def country_moderation_omnibus(participant_diff, outcomes, min_group_n=5):
    rows = []
    for outcome in outcomes:
        diff_col = f"{outcome}_avatar_minus_chat"
        if diff_col not in participant_diff.columns:
            continue
        frame = participant_diff[["country", diff_col]].copy()
        frame[diff_col] = pd.to_numeric(frame[diff_col], errors="coerce")
        frame = frame.dropna()
        counts = frame.groupby("country")[diff_col].count()
        groups = [g for g in COUNTRY_ORDER if counts.get(g, 0) >= min_group_n]
        if len(groups) != 3:
            continue
        frame = frame[frame["country"].isin(groups)].rename(columns={diff_col: "delta"})
        model = smf.ols("delta ~ C(country)", data=frame).fit(cov_type="HC3")
        country_terms = [n for n in model.params.index if n.startswith("C(country)")]
        stat, df, p = _joint_wald(model, country_terms)
        rows.append({
            "outcome": outcome,
            "analysis_design": "within_participant_difference_score",
            "n_participants": len(frame),
            "n_israel": int((frame["country"] == "israel").sum()),
            "n_germany": int((frame["country"] == "germany").sum()),
            "n_china_group": int((frame["country"] == "china").sum()),
            "wald_chi2": stat,
            "df": df,
            "p_value": p,
            "robust_model_r_squared": float(model.rsquared),
        })
    result = pd.DataFrame(rows)
    if not result.empty:
        result["p_holm"] = _holm(result["p_value"])
    return result


def first_importance_country_omnibus(data, min_group_n=5):
    frame = data.loc[data["condition_order"].eq("first")].copy()
    frame["importance_change"] = pd.to_numeric(frame["importance_change"], errors="coerce")
    frame = frame.dropna(subset=["importance_change", "condition", "country"])
    counts = frame.groupby(["country", "condition"])["importance_change"].count()
    if any(counts.get((c, cond), 0) < min_group_n for c in COUNTRY_ORDER for cond in ["Avatar", "Chat"]):
        return pd.DataFrame()
    model = smf.ols(
        'Q("importance_change") ~ C(condition) * C(country)', data=frame
    ).fit(cov_type="HC3")
    terms = [n for n in model.params.index if ":" in n and "C(condition)" in n and "C(country)" in n]
    stat, df, p = _joint_wald(model, terms)
    return pd.DataFrame([{
        "outcome": "importance_change",
        "analysis_design": "first_interaction_condition_by_country_HC3",
        "n_participants": len(frame),
        "n_israel": int((frame["country"] == "israel").sum()),
        "n_germany": int((frame["country"] == "germany").sum()),
        "n_china_group": int((frame["country"] == "china").sum()),
        "wald_chi2": stat,
        "df": df,
        "p_value": p,
        "robust_model_r_squared": float(model.rsquared),
    }])


def country_pairwise_posthoc(participant_diff, outcomes, omnibus_table, min_group_n=5):
    rows = []
    for outcome in outcomes:
        diff_col = f"{outcome}_avatar_minus_chat"
        if diff_col not in participant_diff.columns:
            continue
        frame = participant_diff[["country", diff_col]].copy()
        frame[diff_col] = pd.to_numeric(frame[diff_col], errors="coerce")
        frame = frame.dropna()
        for a_name, b_name in combinations(COUNTRY_ORDER, 2):
            a = frame.loc[frame["country"].eq(a_name), diff_col]
            b = frame.loc[frame["country"].eq(b_name), diff_col]
            if len(a) < min_group_n or len(b) < min_group_n:
                continue
            t_stat, p_value = stats.ttest_ind(a, b, equal_var=False)
            difference, ci_low, ci_high, welch_df = _welch_difference_ci(a, b)
            d = _cohens_d_independent(a, b)
            correction = 1 - 3/(4*(len(a)+len(b))-9)
            g = d*correction if np.isfinite(d) else np.nan
            try:
                u_stat, u_p = stats.mannwhitneyu(a, b, alternative="two-sided")
            except Exception:
                u_stat = u_p = np.nan
            rows.append({
                "outcome": outcome,
                "analysis_design": "within_participant_difference_score",
                "country_a": a_name,
                "country_b": b_name,
                "n_a": len(a),
                "n_b": len(b),
                "mean_delta_a": a.mean(),
                "mean_delta_b": b.mean(),
                "difference_of_deltas_a_minus_b": difference,
                "ci95_low": ci_low,
                "ci95_high": ci_high,
                "welch_t": t_stat,
                "welch_df": welch_df,
                "p_value": p_value,
                "cohens_d": d,
                "hedges_g": g,
                "mann_whitney_u": u_stat,
                "mann_whitney_p": u_p,
            })
    result = pd.DataFrame(rows)
    if result.empty:
        return result
    result["p_holm"] = np.nan
    result["mann_whitney_p_holm"] = np.nan
    for outcome, idx in result.groupby("outcome").groups.items():
        result.loc[idx, "p_holm"] = _holm(result.loc[idx, "p_value"]).values
        result.loc[idx, "mann_whitney_p_holm"] = _holm(result.loc[idx, "mann_whitney_p"]).values
    omni_map = omnibus_table.set_index("outcome")["p_holm"].to_dict() if not omnibus_table.empty else {}
    result["omnibus_p_holm"] = result["outcome"].map(omni_map)
    result["posthoc_interpretation_enabled"] = result["omnibus_p_holm"].lt(0.05)
    return result


def first_importance_country_pairwise(data, omnibus_table, min_group_n=5):
    """Pairwise cohort contrasts of the first-interaction condition effect."""
    frame_all = data.loc[data["condition_order"].eq("first")].copy()
    frame_all["importance_change"] = pd.to_numeric(frame_all["importance_change"], errors="coerce")
    frame_all = frame_all.dropna(subset=["importance_change", "condition", "country"])
    rows = []
    for a_name, b_name in combinations(COUNTRY_ORDER, 2):
        frame = frame_all.loc[frame_all["country"].isin([a_name, b_name])].copy()
        cell_counts = frame.groupby(["country", "condition"])["importance_change"].count()
        if any(cell_counts.get((c, cond), 0) < min_group_n for c in [a_name, b_name] for cond in ["Avatar", "Chat"]):
            continue
        formula = (
            'Q("importance_change") ~ '
            'C(condition, Treatment(reference="Chat")) * '
            f'C(country, Treatment(reference="{b_name}"))'
        )
        fit = smf.ols(formula, data=frame).fit(cov_type="HC3")
        interaction_terms = [n for n in fit.params.index if ":" in n]
        stat, df, p = _joint_wald(fit, interaction_terms)
        a_avatar = frame.loc[(frame["country"] == a_name) & (frame["condition"] == "Avatar"), "importance_change"]
        a_chat = frame.loc[(frame["country"] == a_name) & (frame["condition"] == "Chat"), "importance_change"]
        b_avatar = frame.loc[(frame["country"] == b_name) & (frame["condition"] == "Avatar"), "importance_change"]
        b_chat = frame.loc[(frame["country"] == b_name) & (frame["condition"] == "Chat"), "importance_change"]
        effect_a = float(a_avatar.mean() - a_chat.mean())
        effect_b = float(b_avatar.mean() - b_chat.mean())
        interaction_term = interaction_terms[0] if len(interaction_terms) == 1 else None
        if interaction_term is not None:
            coef = float(fit.params[interaction_term])
            ci = fit.conf_int().loc[interaction_term]
            # With b as reference country, the interaction coefficient is effect_a - effect_b.
            diff_effect = coef
            ci_low, ci_high = float(ci.iloc[0]), float(ci.iloc[1])
        else:
            diff_effect = effect_a - effect_b
            ci_low = ci_high = np.nan
        rows.append({
            "outcome": "importance_change",
            "analysis_design": "first_interaction_difference_in_condition_effects_HC3",
            "country_a": a_name,
            "country_b": b_name,
            "n_a": int((frame["country"] == a_name).sum()),
            "n_b": int((frame["country"] == b_name).sum()),
            "mean_delta_a": effect_a,
            "mean_delta_b": effect_b,
            "difference_of_deltas_a_minus_b": diff_effect,
            "ci95_low": ci_low,
            "ci95_high": ci_high,
            "welch_t": np.nan,
            "welch_df": np.nan,
            "p_value": p,
            "wald_chi2": stat,
            "cohens_d": np.nan,
            "hedges_g": np.nan,
            "mann_whitney_u": np.nan,
            "mann_whitney_p": np.nan,
        })
    result = pd.DataFrame(rows)
    if result.empty:
        return result
    result["p_holm"] = _holm(result["p_value"])
    omni_map = omnibus_table.set_index("outcome")["p_holm"].to_dict() if not omnibus_table.empty else {}
    result["omnibus_p_holm"] = result["outcome"].map(omni_map)
    result["posthoc_interpretation_enabled"] = result["omnibus_p_holm"].lt(0.05)
    return result


def first_importance_tests_by_country(data, min_group_n=5):
    rows = []
    for country, subset in data.loc[data["condition_order"].eq("first")].groupby("country"):
        res = first_interaction_importance_test(subset, min_group_n)
        if not res.empty:
            res.insert(0, "country", country)
            rows.append(res)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


# Paired-difference dataset intentionally excludes importance_change because
# importance is not a valid within-participant Avatar-minus-Chat contrast.
participant_differences = hoa.build_paired_difference_dataset(
    analysis,
    PAIRED_PRIMARY_OUTCOMES + SECONDARY_BEHAVIOR_OUTCOMES + EXPLORATORY_BEHAVIOR_OUTCOMES,
    participant_columns=["country", "gender_normalized"],
)

country_omnibus_primary_paired = country_moderation_omnibus(
    participant_differences, PAIRED_PRIMARY_OUTCOMES, MIN_GROUP_N
)
importance_country_omnibus = first_importance_country_omnibus(analysis, MIN_GROUP_N)
country_omnibus_primary = pd.concat(
    [country_omnibus_primary_paired, importance_country_omnibus],
    ignore_index=True,
    sort=False,
)
if not country_omnibus_primary.empty:
    country_omnibus_primary["p_holm"] = _holm(country_omnibus_primary["p_value"])

country_pairwise_primary_paired = country_pairwise_posthoc(
    participant_differences,
    PAIRED_PRIMARY_OUTCOMES,
    country_omnibus_primary,
    MIN_GROUP_N,
)
importance_country_pairwise = first_importance_country_pairwise(
    analysis, country_omnibus_primary, MIN_GROUP_N
)
country_pairwise_primary = pd.concat(
    [country_pairwise_primary_paired, importance_country_pairwise],
    ignore_index=True,
    sort=False,
)

# Within-cohort condition effects: paired for the three character/interface-specific
# outcomes, independent first-interaction-only for importance.
country_paired_primary_paired = hoa.paired_tests_by_country(
    analysis, PAIRED_PRIMARY_OUTCOMES, MIN_GROUP_N
)
if not country_paired_primary_paired.empty:
    country_paired_primary_paired["analysis_design"] = "within_participant_paired"
    country_paired_primary_paired["n_avatar"] = country_paired_primary_paired["n_pairs"]
    country_paired_primary_paired["n_chat"] = country_paired_primary_paired["n_pairs"]

importance_country_condition = first_importance_tests_by_country(analysis, MIN_GROUP_N)
country_paired_primary = pd.concat(
    [country_paired_primary_paired, importance_country_condition],
    ignore_index=True,
    sort=False,
)
if not country_paired_primary.empty:
    country_paired_primary["p_holm_within_country_family"] = np.nan
    for country, idx in country_paired_primary.groupby("country").groups.items():
        country_paired_primary.loc[idx, "p_holm_within_country_family"] = _holm(
            country_paired_primary.loc[idx, "p_value"]
        ).values
    country_paired_primary["p_holm_all_12_subgroup_tests"] = _holm(
        country_paired_primary["p_value"]
    )

country_descriptives_primary = pd.concat([
    hoa.descriptive_by_country_condition(analysis, PAIRED_PRIMARY_OUTCOMES),
    hoa.descriptive_by_country_condition(
        analysis.loc[analysis["condition_order"].eq("first")],
        ["importance_change"],
    ),
], ignore_index=True, sort=False)

print("OMNIBUS COUNTRY MODERATION - PRIMARY OUTCOMES")
display(country_omnibus_primary.sort_values("p_holm", na_position="last"))

print("POST-HOC COUNTRY PAIRS ON CONDITION EFFECT")
display(country_pairwise_primary.sort_values(["outcome", "p_holm"], na_position="last"))

print("WITHIN-COUNTRY CONDITION EFFECTS - DESCRIPTIVE/SUPPLEMENTARY")
display(country_paired_primary.sort_values(["country", "outcome"]))


OMNIBUS COUNTRY MODERATION - PRIMARY OUTCOMES


,outcome,analysis_design,n_participants,n_israel,n_germany,n_china_group,wald_chi2,df,p_value,robust_model_r_squared,p_holm
0,character_connection,within_participant_difference_score,146,49,43,54,9.344596,2,0.009351,0.049904,0.037403
1,satisfaction,within_participant_difference_score,146,49,43,54,1.101390,2,0.576549,0.006555,1.000000
2,knowledge_accuracy,within_participant_difference_score,146,49,43,54,1.281053,2,0.527015,0.008695,1.000000
3,importance_change,first_interaction_condition_by_country_HC3,146,49,43,54,1.105583,2,0.575341,0.027799,1.000000


POST-HOC COUNTRY PAIRS ON CONDITION EFFECT


,outcome,analysis_design,country_a,country_b,n_a,n_b,mean_delta_a,mean_delta_b,difference_of_deltas_a_minus_b,ci95_low,...,p_value,cohens_d,hedges_g,mann_whitney_u,mann_whitney_p,p_holm,mann_whitney_p_holm,omnibus_p_holm,posthoc_interpretation_enabled,wald_chi2
1,character_connection,within_participant_difference_score,israel,china,49,54,0.479592,1.018519e-01,0.377740,0.092494,...,0.010130,0.533311,0.529341,1654.0,0.013411,0.030391,0.040233,0.037403,True,NaN
2,character_connection,within_participant_difference_score,germany,china,43,54,0.418605,1.018519e-01,0.316753,0.024799,...,0.033920,0.467268,0.463569,1439.5,0.024140,0.067840,0.048281,0.037403,True,NaN
0,character_connection,within_participant_difference_score,israel,germany,49,43,0.479592,4.186047e-01,0.060987,-0.296185,...,0.735207,0.070713,0.070122,1087.0,0.788760,0.735207,0.788760,0.037403,True,NaN
9,importance_change,first_interaction_difference_in_condition_effe...,israel,germany,49,43,0.121667,9.956710e-02,0.022100,-0.372388,...,0.912569,NaN,NaN,NaN,NaN,1.000000,NaN,1.000000,False,0.012056
10,importance_change,first_interaction_difference_in_condition_effe...,israel,china,49,54,0.121667,-7.692308e-02,0.198590,-0.217135,...,0.349137,NaN,NaN,NaN,NaN,1.000000,NaN,1.000000,False,0.876593
11,importance_change,first_interaction_difference_in_condition_effe...,germany,china,43,54,0.099567,-7.692308e-02,0.176490,-0.210152,...,0.370967,NaN,NaN,NaN,NaN,1.000000,NaN,1.000000,False,0.800422
6,knowledge_accuracy,within_participant_difference_score,israel,germany,49,43,0.057143,4.651163e-03,0.052492,-0.065310,...,0.378191,0.186116,0.184561,1136.5,0.514025,0.885875,1.000000,1.000000,False,NaN
7,knowledge_accuracy,within_participant_difference_score,israel,china,49,54,0.057143,-4.111937e-18,0.057143,-0.050609,...,0.295292,0.207225,0.205682,1510.5,0.212624,0.885875,0.637872,1.000000,False,NaN
8,knowledge_accuracy,within_participant_difference_score,germany,china,43,54,0.004651,-4.111937e-18,0.004651,-0.112386,...,0.937232,0.016233,0.016105,1235.5,0.584245,0.937232,1.000000,1.000000,False,NaN
3,satisfaction,within_participant_difference_score,israel,germany,49,43,0.414966,4.186047e-01,-0.003639,-0.419533,...,0.986170,-0.003618,-0.003588,1007.0,0.715620,1.000000,0.882777,1.000000,False,NaN


WITHIN-COUNTRY CONDITION EFFECTS - DESCRIPTIVE/SUPPLEMENTARY


,country,outcome,n_pairs,avatar_mean,avatar_sd,chat_mean,chat_sd,mean_difference_avatar_minus_chat,difference_ci95_low,difference_ci95_high,...,n_chat,welch_t,welch_df,cohens_d,mann_whitney_u,mann_whitney_p,rank_test,rank_test_p,p_holm_within_country_family,p_holm_all_12_subgroup_tests
0,china,character_connection,54.0,3.629630,0.807893,3.527778,0.865571,1.018519e-01,-0.036868,0.240572,...,54,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.440264,1.000000
9,china,importance_change,NaN,0.000000,0.272166,0.076923,0.688365,-7.692308e-02,-0.371134,0.217288,...,26,-0.532466,32.157332,-0.149071,350.0,0.739411,Mann-Whitney U,0.739411,1.000000,1.000000
2,china,knowledge_accuracy,54.0,0.581481,0.224091,0.581481,0.233976,-4.111937e-18,-0.076469,0.076469,...,54,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,1.000000
1,china,satisfaction,54.0,3.666667,0.856594,3.401235,0.807147,2.654321e-01,0.070713,0.460151,...,54,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.033937,0.071156
3,germany,character_connection,43.0,3.511628,0.922525,3.093023,1.013339,4.186047e-01,0.158656,0.678553,...,43,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.009109,0.025049
10,germany,importance_change,NaN,0.190476,0.511766,0.090909,0.294245,9.956710e-02,-0.161469,0.360603,...,22,0.777320,31.615412,0.240005,244.0,0.584942,Mann-Whitney U,0.584942,0.885496,1.000000
5,germany,knowledge_accuracy,43.0,0.660465,0.211727,0.655814,0.210783,4.651163e-03,-0.085938,0.095240,...,43,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.917968,1.000000
4,germany,satisfaction,43.0,3.860465,1.024134,3.441860,0.893009,4.186047e-01,0.119025,0.718185,...,43,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.021904,0.071156
6,israel,character_connection,49.0,3.734694,0.884485,3.255102,0.842383,4.795918e-01,0.227479,0.731705,...,49,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.001510,0.004531
11,israel,importance_change,NaN,0.041667,0.550033,-0.080000,0.493288,1.216667e-01,-0.179191,0.422525,...,25,0.814029,45.967250,0.233155,323.0,0.521981,Mann-Whitney U,0.521981,0.419826,1.000000


# Part IV - Adjusted Repeated-Measures Sensitivity Models

Fit adjusted repeated-measures sensitivity models for the primary outcomes.


In [20]:
def fit_primary_adjusted_models(data, outcomes, min_group_n=5):
    coef_frames, interaction_rows = [], []
    for outcome in outcomes:
        if outcome not in data.columns:
            continue

        if outcome == "importance_change":
            # Global attitude outcome: only the first interaction is causally attributable
            # to its assigned interface without prior-condition carryover.
            frame = data.loc[data["condition_order"].eq("first")].copy()
            frame[outcome] = pd.to_numeric(frame[outcome], errors="coerce")
            required = [outcome, "condition", "country", "character", "user_id"]
            frame = frame.dropna(subset=required)
            if len(frame) < max(20, min_group_n * 2):
                continue
            formula = f'Q("{outcome}") ~ C(condition) * C(country) + C(character)'
            fit = smf.ols(formula, frame).fit(cov_type="HC3")
            model_kind = "first_interaction_HC3_OLS"
        else:
            frame = data.copy()
            frame[outcome] = pd.to_numeric(frame[outcome], errors="coerce")
            required = [outcome, "condition", "country", "character", "user_id"]
            frame = frame.dropna(subset=required)
            if len(frame) < max(20, min_group_n * 2):
                continue
            terms = ["C(condition) * C(country)", "C(character)"]
            if "condition_position" in frame.columns and frame["condition_position"].notna().sum() >= 10:
                terms.append("condition_position")
            formula = f'Q("{outcome}") ~ ' + " + ".join(terms)
            fit = smf.ols(formula, frame).fit(
                cov_type="cluster", cov_kwds={"groups": frame["user_id"]}
            )
            model_kind = "participant_clustered_OLS"

        coef = hoa.flatten_model(fit, f"{model_kind}: {formula}", outcome)
        coef["model_kind"] = model_kind
        coef_frames.append(coef)

        interaction_terms = [
            name for name in fit.params.index
            if ":" in name and "C(condition)" in name and "C(country)" in name
        ]
        stat, df, p = _joint_wald(fit, interaction_terms)
        interaction_rows.append({
            "outcome": outcome,
            "model_kind": model_kind,
            "n_obs": int(fit.nobs),
            "n_participants": frame["user_id"].nunique(),
            "interaction_df": df,
            "wald_chi2": stat,
            "p_value": p,
            "formula": formula,
        })

    coefs = pd.concat(coef_frames, ignore_index=True) if coef_frames else pd.DataFrame()
    interactions = pd.DataFrame(interaction_rows)
    if not interactions.empty:
        interactions["p_holm"] = _holm(interactions["p_value"])
    return coefs, interactions


primary_adjusted_coefficients, primary_adjusted_interactions = (
    fit_primary_adjusted_models(analysis, PRIMARY_OUTCOMES, MIN_GROUP_N)
)

print("TARGETED CONDITION × COUNTRY TESTS - ADJUSTED SENSITIVITY MODELS")
display(primary_adjusted_interactions.sort_values("p_holm", na_position="last"))

print("Full coefficient tables are supplementary; raw p-values only:")
display(primary_adjusted_coefficients.head(40))


TARGETED CONDITION × COUNTRY TESTS - ADJUSTED SENSITIVITY MODELS


,outcome,model_kind,n_obs,n_participants,interaction_df,wald_chi2,p_value,formula,p_holm
0,character_connection,participant_clustered_OLS,292,146,2,8.400437,0.014992,"Q(""character_connection"") ~ C(condition) * C(c...",0.059969
1,satisfaction,participant_clustered_OLS,292,146,2,1.366149,0.505062,"Q(""satisfaction"") ~ C(condition) * C(country) ...",1.000000
2,knowledge_accuracy,participant_clustered_OLS,292,146,2,1.492165,0.474221,"Q(""knowledge_accuracy"") ~ C(condition) * C(cou...",1.000000
3,importance_change,first_interaction_HC3_OLS,146,146,2,1.360940,0.506379,"Q(""importance_change"") ~ C(condition) * C(coun...",1.000000


Full coefficient tables are supplementary; raw p-values only:


,model,outcome,term,estimate,std_error,statistic,p_value,ci_low,ci_high,n_obs,aic,bic,model_kind
0,"participant_clustered_OLS: Q(""character_connec...",character_connection,Intercept,3.775490,0.149325,25.283686,4.828872e-141,3.482819,4.068162,292,767.448913,796.862943,participant_clustered_OLS
1,"participant_clustered_OLS: Q(""character_connec...",character_connection,C(condition)[T.Chat],-0.102358,0.069296,-1.477119,1.396437e-01,-0.238175,0.033459,292,767.448913,796.862943,participant_clustered_OLS
2,"participant_clustered_OLS: Q(""character_connec...",character_connection,C(country)[T.germany],-0.119186,0.180674,-0.659676,5.094616e-01,-0.473300,0.234928,292,767.448913,796.862943,participant_clustered_OLS
3,"participant_clustered_OLS: Q(""character_connec...",character_connection,C(country)[T.israel],0.103316,0.171086,0.603882,5.459222e-01,-0.232007,0.438639,292,767.448913,796.862943,participant_clustered_OLS
4,"participant_clustered_OLS: Q(""character_connec...",character_connection,C(character)[T.Markel],-0.033221,0.065287,-0.508842,6.108633e-01,-0.161181,0.094740,292,767.448913,796.862943,participant_clustered_OLS
5,"participant_clustered_OLS: Q(""character_connec...",character_connection,C(condition)[T.Chat]:C(country)[T.germany],-0.314384,0.146676,-2.143391,3.208170e-02,-0.601863,-0.026904,292,767.448913,796.862943,participant_clustered_OLS
6,"participant_clustered_OLS: Q(""character_connec...",character_connection,C(condition)[T.Chat]:C(country)[T.israel],-0.374243,0.151799,-2.465392,1.368634e-02,-0.671763,-0.076723,292,767.448913,796.862943,participant_clustered_OLS
7,"participant_clustered_OLS: Q(""character_connec...",character_connection,condition_position,-0.085998,0.062472,-1.376581,1.686420e-01,-0.208442,0.036445,292,767.448913,796.862943,participant_clustered_OLS
8,"participant_clustered_OLS: Q(""satisfaction"") ~...",satisfaction,Intercept,3.589017,0.154716,23.197434,4.833053e-119,3.285779,3.892255,292,782.846409,812.260439,participant_clustered_OLS
9,"participant_clustered_OLS: Q(""satisfaction"") ~...",satisfaction,C(condition)[T.Chat],-0.257545,0.096078,-2.680574,7.349597e-03,-0.445855,-0.069235,292,782.846409,812.260439,participant_clustered_OLS


# Part V - Item-Level Knowledge Robustness Analysis

Evaluate item-level knowledge patterns and robustness to question difficulty.


In [21]:
def fit_publication_knowledge_gee(knowledge_items, analysis):
    design_cols = [
        "user_id", "session_id", "character", "condition", "country", "condition_position"
    ]
    design = analysis[[c for c in design_cols if c in analysis.columns]].drop_duplicates(
        ["user_id", "session_id", "character"]
    )
    frame = knowledge_items.merge(
        design, on=["user_id", "session_id", "character"], how="left"
    )
    frame = frame.dropna(subset=["is_correct", "condition", "country", "stage_id", "user_id"])
    terms = ["C(condition) * C(country)", "C(stage_id)"]
    if "condition_position" in frame.columns and frame["condition_position"].notna().sum() >= 10:
        terms.append("condition_position")
    formula = "is_correct ~ " + " + ".join(terms)
    model = smf.gee(
        formula,
        groups="user_id",
        data=frame,
        family=sm.families.Binomial(),
        cov_struct=sm.cov_struct.Exchangeable(),
    ).fit()
    import warnings
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            category=FutureWarning,
            message="The bic value is computed using the deviance formula.*",
        )
        coefs = hoa.flatten_model(model, f"Binomial GEE: {formula}", "is_correct")
    coefs["odds_ratio"] = np.exp(coefs["estimate"])
    coefs["or_ci_low"] = np.exp(coefs["ci_low"])
    coefs["or_ci_high"] = np.exp(coefs["ci_high"])
    interaction_terms = [
        name for name in model.params.index
        if ":" in name and "C(condition)" in name and "C(country)" in name
    ]
    stat, df, p = _joint_wald(model, interaction_terms)
    interaction = pd.DataFrame([{
        "outcome": "knowledge_item_correct",
        "n_item_rows": len(frame),
        "n_participants": frame["user_id"].nunique(),
        "interaction_df": df,
        "wald_chi2": stat,
        "p_value": p,
        "formula": formula,
    }])
    return coefs, interaction, frame


knowledge_gee_coefficients, knowledge_gee_interaction, knowledge_gee_frame = (
    fit_publication_knowledge_gee(knowledge_items, analysis)
)

print("Participant-level primary knowledge result:")
display(whole_sample_primary.loc[whole_sample_primary["outcome"].eq("knowledge_accuracy")])

print("Item-level GEE - targeted Condition × Country interaction:")
display(knowledge_gee_interaction)

print("Item-level GEE coefficient table (supplementary):")
display(knowledge_gee_coefficients.head(40))


Participant-level primary knowledge result:


,outcome,n_pairs,avatar_mean,avatar_sd,chat_mean,chat_sd,mean_difference_avatar_minus_chat,difference_ci95_low,difference_ci95_high,paired_t,...,n_chat,welch_t,welch_df,cohens_d,mann_whitney_u,mann_whitney_p,rank_test,rank_test_p,rank_test_p_holm,mann_whitney_p_holm
2,knowledge_accuracy,146.0,0.638356,0.221985,0.617808,0.221495,0.020548,-0.025354,0.06645,0.884763,...,146,NaN,NaN,NaN,NaN,NaN,Wilcoxon signed-rank,0.393409,0.786817,NaN


Item-level GEE - targeted Condition × Country interaction:


,outcome,n_item_rows,n_participants,interaction_df,wald_chi2,p_value,formula
0,knowledge_item_correct,1460,146,2,1.567501,0.45669,is_correct ~ C(condition) * C(country) + C(sta...


Item-level GEE coefficient table (supplementary):


,model,outcome,term,estimate,std_error,statistic,p_value,ci_low,ci_high,n_obs,aic,bic,odds_ratio,or_ci_low,or_ci_high
0,Binomial GEE: is_correct ~ C(condition) * C(co...,is_correct,Intercept,1.195039,0.275854,4.332142,1.476655e-05,0.654375,1.735703,1460,1792.697636,-8760.5632,3.303686,1.923940,5.672914
1,Binomial GEE: is_correct ~ C(condition) * C(co...,is_correct,C(condition)[T.Chat],0.006330,0.172772,0.036635,9.707758e-01,-0.332298,0.344957,1460,1792.697636,-8760.5632,1.006350,0.717273,1.411930
2,Binomial GEE: is_correct ~ C(condition) * C(co...,is_correct,C(country)[T.germany],0.366638,0.208650,1.757188,7.888577e-02,-0.042309,0.775585,1460,1792.697636,-8760.5632,1.442875,0.958573,2.171862
3,Binomial GEE: is_correct ~ C(condition) * C(co...,is_correct,C(country)[T.israel],0.500375,0.207835,2.407562,1.605944e-02,0.093026,0.907724,1460,1792.697636,-8760.5632,1.649340,1.097491,2.478676
4,Binomial GEE: is_correct ~ C(condition) * C(co...,is_correct,C(stage_id)[T.ben_lesser_q2],-1.530989,0.272807,-5.611990,2.000133e-08,-2.065681,-0.996297,1460,1792.697636,-8760.5632,0.216322,0.126732,0.369244
5,Binomial GEE: is_correct ~ C(condition) * C(co...,is_correct,C(stage_id)[T.ben_lesser_q3],-0.691246,0.290311,-2.381056,1.726309e-02,-1.260245,-0.122248,1460,1792.697636,-8760.5632,0.500951,0.283585,0.884929
6,Binomial GEE: is_correct ~ C(condition) * C(co...,is_correct,C(stage_id)[T.ben_lesser_q4],-2.151296,0.268204,-8.021131,1.047760e-15,-2.676965,-1.625626,1460,1792.697636,-8760.5632,0.116333,0.068772,0.196788
7,Binomial GEE: is_correct ~ C(condition) * C(co...,is_correct,C(stage_id)[T.ben_lesser_q5],-1.103052,0.283584,-3.889680,1.003766e-04,-1.658867,-0.547237,1460,1792.697636,-8760.5632,0.331857,0.190355,0.578546
8,Binomial GEE: is_correct ~ C(condition) * C(co...,is_correct,C(stage_id)[T.markel_q1],0.549973,0.316306,1.738738,8.208086e-02,-0.069975,1.169920,1460,1792.697636,-8760.5632,1.733206,0.932417,3.221736
9,Binomial GEE: is_correct ~ C(condition) * C(co...,is_correct,C(stage_id)[T.markel_q2],-1.602899,0.274696,-5.835183,5.373167e-09,-2.141292,-1.064505,1460,1792.697636,-8760.5632,0.201312,0.117503,0.344898


### Knowledge-question difficulty

Summarize item difficulty for the character-specific knowledge questions.


In [22]:
# Per-question knowledge difficulty analysis

# ------------------------------------------------------------
# 1. Overall correct-answer rate for every knowledge question
# ------------------------------------------------------------

question_summary = (
    knowledge_gee_frame
    .groupby(["character", "stage_id"], observed=True)
    .agg(
        n=("is_correct", "size"),
        n_correct=("is_correct", "sum"),
        accuracy=("is_correct", "mean"),
    )
    .reset_index()
)

question_summary["percent_correct"] = (
    100 * question_summary["accuracy"]
)

question_summary["majority_correct"] = (
    question_summary["percent_correct"] > 50
)

print("Correct-answer rate by question:")
display(
    question_summary[
        [
            "character",
            "stage_id",
            "n",
            "n_correct",
            "percent_correct",
            "majority_correct",
        ]
    ]
    .sort_values(
        ["character", "percent_correct"],
        ascending=[True, False]
    )
)


# ------------------------------------------------------------
# 2. Same questions separately for Avatar and Chat
# ------------------------------------------------------------

question_condition = (
    knowledge_gee_frame
    .groupby(
        ["character", "stage_id", "condition"],
        observed=True
    )
    .agg(
        n=("is_correct", "size"),
        accuracy=("is_correct", "mean"),
    )
    .reset_index()
)

question_condition["percent_correct"] = (
    100 * question_condition["accuracy"]
)

question_condition_wide = (
    question_condition
    .pivot(
        index=["character", "stage_id"],
        columns="condition",
        values="percent_correct"
    )
    .reset_index()
)

question_condition_wide["Avatar_minus_Chat_pp"] = (
    question_condition_wide["Avatar"]
    - question_condition_wide["Chat"]
)

print("\nCorrect-answer percentage by question and condition:")
display(question_condition_wide)


# ------------------------------------------------------------
# 3. Omnibus statistical test:
#    do questions differ in difficulty overall?
# ------------------------------------------------------------

terms = [
    "C(condition) * C(country)",
    "C(stage_id)"
]

if (
    "condition_position" in knowledge_gee_frame.columns
    and knowledge_gee_frame["condition_position"].notna().sum() >= 10
):
    terms.append("condition_position")

question_formula = "is_correct ~ " + " + ".join(terms)

question_model = smf.gee(
    question_formula,
    groups="user_id",
    data=knowledge_gee_frame,
    family=sm.families.Binomial(),
    cov_struct=sm.cov_struct.Exchangeable(),
).fit()

question_terms = [
    name
    for name in question_model.params.index
    if "C(stage_id)" in name
]

wald_stat, df, p_question = _joint_wald(
    question_model,
    question_terms
)

print(
    f"\nOmnibus question-difficulty test: "
    f"Wald chi2({df})={wald_stat:.3f}, "
    f"p={p_question:.6f}"
)


Correct-answer rate by question:


,character,stage_id,n,n_correct,percent_correct,majority_correct
0,Ben Lesser,ben_lesser_q1,146,122,83.561644,True
2,Ben Lesser,ben_lesser_q3,146,105,71.917808,True
4,Ben Lesser,ben_lesser_q5,146,92,63.013699,True
1,Ben Lesser,ben_lesser_q2,146,77,52.739726,True
3,Ben Lesser,ben_lesser_q4,146,55,37.671233,False
5,Markel,markel_q1,146,131,89.726027,True
8,Markel,markel_q4,146,98,67.123288,True
9,Markel,markel_q5,146,89,60.958904,True
6,Markel,markel_q2,146,74,50.684932,True
7,Markel,markel_q3,146,74,50.684932,True



Correct-answer percentage by question and condition:


condition,character,stage_id,Avatar,Chat,Avatar_minus_Chat_pp
0,Ben Lesser,ben_lesser_q1,85.526316,81.428571,4.097744
1,Ben Lesser,ben_lesser_q2,55.263158,50.000000,5.263158
2,Ben Lesser,ben_lesser_q3,75.000000,68.571429,6.428571
3,Ben Lesser,ben_lesser_q4,38.157895,37.142857,1.015038
4,Ben Lesser,ben_lesser_q5,71.052632,54.285714,16.766917
5,Markel,markel_q1,84.285714,94.736842,-10.451128
6,Markel,markel_q2,51.428571,50.000000,1.428571
7,Markel,markel_q3,54.285714,47.368421,6.917293
8,Markel,markel_q4,67.142857,67.105263,0.037594
9,Markel,markel_q5,55.714286,65.789474,-10.075188



Omnibus question-difficulty test: Wald chi2(9)=119.193, p=0.000000


### Avatar vs. Chat on the four hardest knowledge items

Repeat the condition comparison using the four hardest knowledge items.


In [23]:
# ============================================================
# Avatar vs Chat on the four hardest knowledge questions
# Overall sample + China only
# ============================================================

# Select the four hardest items based on pooled accuracy
# (ignoring condition when defining difficulty)
hard_items = (
    question_summary
    .nsmallest(4, "accuracy")["stage_id"]
    .tolist()
)

print("Four hardest items:")
display(
    question_summary[
        question_summary["stage_id"].isin(hard_items)
    ][
        ["character", "stage_id", "n", "n_correct", "percent_correct"]
    ]
    .sort_values("percent_correct")
)

# Keep only hard items
hard = knowledge_gee_frame[
    knowledge_gee_frame["stage_id"].isin(hard_items)
].copy()

# ------------------------------------------------------------
# Participant-level hard-item accuracy by condition
# Each participant should contribute hard items to both conditions
# ------------------------------------------------------------

hard_participant = (
    hard
    .groupby(
        ["user_id", "country", "condition"],
        observed=True
    )
    .agg(
        hard_accuracy=("is_correct", "mean"),
        n_hard_items=("is_correct", "size"),
    )
    .reset_index()
)

hard_wide = (
    hard_participant
    .pivot_table(
        index=["user_id", "country"],
        columns="condition",
        values="hard_accuracy"
    )
    .dropna(subset=["Avatar", "Chat"])
    .reset_index()
)

hard_wide["avatar_minus_chat"] = (
    hard_wide["Avatar"] - hard_wide["Chat"]
)

# ------------------------------------------------------------
# 1. Overall sample
# ------------------------------------------------------------

t_all, p_all = stats.ttest_rel(
    hard_wide["Avatar"],
    hard_wide["Chat"]
)

try:
    w_all, p_w_all = stats.wilcoxon(
        hard_wide["Avatar"],
        hard_wide["Chat"]
    )
except ValueError:
    w_all, p_w_all = np.nan, np.nan

print("\nALL PARTICIPANTS - HARD QUESTIONS")
print(f"N={len(hard_wide)}")
print(
    f"Avatar accuracy={hard_wide['Avatar'].mean():.3f} "
    f"({100*hard_wide['Avatar'].mean():.1f}%)"
)
print(
    f"Chat accuracy={hard_wide['Chat'].mean():.3f} "
    f"({100*hard_wide['Chat'].mean():.1f}%)"
)
print(
    f"Avatar-Chat difference="
    f"{100*hard_wide['avatar_minus_chat'].mean():.1f} percentage points"
)
print(
    f"Paired t-test: t={t_all:.3f}, p={p_all:.4f}"
)
print(
    f"Wilcoxon: W={w_all:.3f}, p={p_w_all:.4f}"
)


# ------------------------------------------------------------
# 2. China only
# ------------------------------------------------------------

china_hard = hard_wide[
    hard_wide["country"].str.lower() == "china"
].copy()

t_china, p_china = stats.ttest_rel(
    china_hard["Avatar"],
    china_hard["Chat"]
)

try:
    w_china, p_w_china = stats.wilcoxon(
        china_hard["Avatar"],
        china_hard["Chat"]
    )
except ValueError:
    w_china, p_w_china = np.nan, np.nan

print("\nCHINA ONLY - HARD QUESTIONS")
print(f"N={len(china_hard)}")
print(
    f"Avatar accuracy={china_hard['Avatar'].mean():.3f} "
    f"({100*china_hard['Avatar'].mean():.1f}%)"
)
print(
    f"Chat accuracy={china_hard['Chat'].mean():.3f} "
    f"({100*china_hard['Chat'].mean():.1f}%)"
)
print(
    f"Avatar-Chat difference="
    f"{100*china_hard['avatar_minus_chat'].mean():.1f} percentage points"
)
print(
    f"Paired t-test: t={t_china:.3f}, p={p_china:.4f}"
)
print(
    f"Wilcoxon: W={w_china:.3f}, p={p_w_china:.4f}"
)


Four hardest items:


,character,stage_id,n,n_correct,percent_correct
3,Ben Lesser,ben_lesser_q4,146,55,37.671233
6,Markel,markel_q2,146,74,50.684932
7,Markel,markel_q3,146,74,50.684932
1,Ben Lesser,ben_lesser_q2,146,77,52.739726



ALL PARTICIPANTS - HARD QUESTIONS
N=146
Avatar accuracy=0.497 (49.7%)
Chat accuracy=0.462 (46.2%)
Avatar-Chat difference=3.4 percentage points
Paired t-test: t=0.795, p=0.4282
Wilcoxon: W=1919.500, p=0.3645

CHINA ONLY - HARD QUESTIONS
N=54
Avatar accuracy=0.463 (46.3%)
Chat accuracy=0.417 (41.7%)
Avatar-Chat difference=4.6 percentage points
Paired t-test: t=0.742, p=0.4612
Wilcoxon: W=197.000, p=0.4315


### Importance change among participants with room to improve

Re-estimate importance change among participants whose baseline score allowed upward movement.


In [24]:
# ============================================================
# Importance analysis among participants with baseline <= 4
# FIRST INTERACTION ONLY (avoids carryover from interaction 1 into interaction 2)
# ============================================================

importance_room = analysis.loc[
    analysis["initial_importance"].le(4) & analysis["condition_order"].eq("first"),
    [
        "user_id", "session_id", "country", "condition", "character",
        "initial_importance", "post_importance", "importance_change",
    ],
].dropna().copy()

participants_room = importance_room[
    ["user_id", "session_id", "country", "initial_importance"]
].drop_duplicates(["user_id", "session_id"])

print(f"Participants with baseline importance <= 4 and first interaction available: N={len(participants_room)}")
print("\nParticipants by country:")
display(
    participants_room["country"].value_counts().rename_axis("country").reset_index(name="n")
)

condition_summary = (
    importance_room
    .groupby("condition")
    .agg(
        n=("importance_change", "size"),
        baseline_mean=("initial_importance", "mean"),
        post_mean=("post_importance", "mean"),
        mean_change=("importance_change", "mean"),
        sd_change=("importance_change", "std"),
        median_change=("importance_change", "median"),
        prop_increased=("importance_change", lambda x: (x > 0).mean()),
    )
    .reset_index()
)
condition_summary["percent_increased"] = 100 * condition_summary["prop_increased"]
print("\nFirst-interaction importance among participants with room to improve:")
display(condition_summary)

# Within each first-assigned condition: did the post-1 rating differ from baseline?
one_sample_results = []
for condition in ["Avatar", "Chat"]:
    x = importance_room.loc[importance_room["condition"].eq(condition), "importance_change"]
    t_stat, p_t = stats.ttest_1samp(x, popmean=0)
    try:
        w_stat, p_w = stats.wilcoxon(x, alternative="two-sided")
    except ValueError:
        w_stat, p_w = np.nan, np.nan
    one_sample_results.append({
        "condition": condition,
        "n": len(x),
        "mean_change": x.mean(),
        "t": t_stat,
        "p_t": p_t,
        "wilcoxon_W": w_stat,
        "p_wilcoxon": p_w,
    })
one_sample_results = pd.DataFrame(one_sample_results)
print("\nChange from baseline after the FIRST interaction within each condition:")
display(one_sample_results)

# Between groups: Avatar-first versus Chat-first.
avatar_room = importance_room.loc[importance_room["condition"].eq("Avatar"), "importance_change"]
chat_room = importance_room.loc[importance_room["condition"].eq("Chat"), "importance_change"]
welch_room = stats.ttest_ind(avatar_room, chat_room, equal_var=False)
diff_room, ci_low_room, ci_high_room, df_room = _welch_difference_ci(avatar_room, chat_room)
u_room, p_u_room = stats.mannwhitneyu(avatar_room, chat_room, alternative="two-sided")

first_only_room_comparison = pd.DataFrame([{
    "n_avatar_first": len(avatar_room),
    "n_chat_first": len(chat_room),
    "avatar_first_mean_change": avatar_room.mean(),
    "chat_first_mean_change": chat_room.mean(),
    "difference_avatar_minus_chat": diff_room,
    "ci95_low": ci_low_room,
    "ci95_high": ci_high_room,
    "welch_t": welch_room.statistic,
    "welch_df": df_room,
    "p_value": welch_room.pvalue,
    "cohens_d": _cohens_d_independent(avatar_room, chat_room),
    "mann_whitney_u": u_room,
    "mann_whitney_p": p_u_room,
}])

print("\nAvatar-first vs Chat-first among baseline <= 4:")
display(first_only_room_comparison)

country_condition_summary = (
    importance_room
    .groupby(["country", "condition"])
    .agg(
        n=("importance_change", "size"),
        baseline_mean=("initial_importance", "mean"),
        post_mean=("post_importance", "mean"),
        mean_change=("importance_change", "mean"),
        percent_increased=("importance_change", lambda x: 100 * (x > 0).mean()),
    )
    .reset_index()
)
print("\nFirst-interaction results by country × condition:")
display(country_condition_summary)


Participants with baseline importance <= 4 and first interaction available: N=63

Participants by country:


,country,n
0,china,33
1,germany,18
2,israel,12



First-interaction importance among participants with room to improve:


,condition,n,baseline_mean,post_mean,mean_change,sd_change,median_change,prop_increased,percent_increased
0,Avatar,30,3.700000,3.966667,0.266667,0.583292,0.0,0.200000,20.000000
1,Chat,33,3.606061,3.878788,0.272727,0.574060,0.0,0.272727,27.272727



Change from baseline after the FIRST interaction within each condition:


,condition,n,mean_change,t,p_t,wilcoxon_W,p_wilcoxon
0,Avatar,30,0.266667,2.504051,0.018155,0.0,0.023141
1,Chat,33,0.272727,2.729153,0.010231,5.0,0.012555



Avatar-first vs Chat-first among baseline <= 4:


,n_avatar_first,n_chat_first,avatar_first_mean_change,chat_first_mean_change,difference_avatar_minus_chat,ci95_low,ci95_high,welch_t,welch_df,p_value,cohens_d,mann_whitney_u,mann_whitney_p
0,30,33,0.266667,0.272727,-0.006061,-0.298158,0.286036,-0.0415,60.233062,0.967034,-0.010477,477.0,0.751392



First-interaction results by country × condition:


,country,condition,n,baseline_mean,post_mean,mean_change,percent_increased
0,china,Avatar,16,3.750000,3.812500,0.062500,6.250000
1,china,Chat,17,3.529412,3.823529,0.294118,29.411765
2,germany,Avatar,8,3.750000,4.250000,0.500000,37.500000
3,germany,Chat,10,3.700000,3.900000,0.200000,20.000000
4,israel,Avatar,6,3.500000,4.000000,0.500000,33.333333
5,israel,Chat,6,3.666667,4.000000,0.333333,33.333333


# Part VI - Voice and Input Modality

Analyze spoken-input uptake, condition differences, and cohort heterogeneity in input modality.


In [25]:
voice_descriptives = (
    analysis.groupby(["country", "condition"], dropna=False)
    .agg(
        n_participants=("user_id", "nunique"),
        mean_voice_message_rate=("voice_message_rate", "mean"),
        sd_voice_message_rate=("voice_message_rate", "std"),
        any_voice_rate=("used_voice", "mean"),
        mean_voice_messages=("n_voice_messages", "mean"),
        mean_typed_messages=("n_typed_messages", "mean"),
    )
    .reset_index()
)

mcnemar_voice_overall = hoa.mcnemar_condition_test(analysis, "used_voice")
mcnemar_voice_by_country_frames = []
for country in COUNTRY_ORDER:
    result = hoa.mcnemar_condition_test(
        analysis.loc[analysis["country"].eq(country)], "used_voice"
    )
    if not result.empty:
        result.insert(0, "country", country)
        mcnemar_voice_by_country_frames.append(result)
mcnemar_voice_by_country = (
    pd.concat(mcnemar_voice_by_country_frames, ignore_index=True)
    if mcnemar_voice_by_country_frames else pd.DataFrame()
)
if not mcnemar_voice_by_country.empty:
    mcnemar_voice_by_country["p_holm_across_countries"] = _holm(
        mcnemar_voice_by_country["p_value"]
    )

# Direct between-country comparison of the paired binary change in any voice use.
analysis_voice_binary = analysis.copy()
analysis_voice_binary["used_voice_numeric"] = analysis_voice_binary["used_voice"].astype(int)
voice_binary_differences = hoa.build_paired_difference_dataset(
    analysis_voice_binary,
    ["used_voice_numeric"],
    participant_columns=["country"],
)
voice_any_country_omnibus = country_moderation_omnibus(
    voice_binary_differences, ["used_voice_numeric"], MIN_GROUP_N
)
voice_any_country_pairwise = country_pairwise_posthoc(
    voice_binary_differences,
    ["used_voice_numeric"],
    voice_any_country_omnibus,
    MIN_GROUP_N,
)

voice_country_omnibus = country_moderation_omnibus(
    participant_differences, ["voice_message_rate"], MIN_GROUP_N
)
voice_country_pairwise = country_pairwise_posthoc(
    participant_differences, ["voice_message_rate"], voice_country_omnibus, MIN_GROUP_N
)

condition_mode_table = pd.crosstab(
    analysis["condition"], analysis["input_mode_pattern"], margins=True, dropna=False
).reset_index()
country_condition_mode_table = pd.crosstab(
    [analysis["country"], analysis["condition"]],
    analysis["input_mode_pattern"],
    margins=True,
    dropna=False,
).reset_index()
mode_inference_note = pd.DataFrame([{
    "test": "ordinary_chi_square_condition_or_country_by_input_mode_pattern",
    "performed": False,
    "reason": "Not used for inference because participant observations are repeated across conditions; paired McNemar and participant-level change-score tests are used instead.",
}])

print("Voice descriptives:")
display(voice_descriptives)
print("Overall paired McNemar:")
display(mcnemar_voice_overall)
print("Within-country paired McNemar with Holm across three countries:")
display(mcnemar_voice_by_country)
print("Direct country moderation of any-voice paired change (−1/0/+1):")
display(voice_any_country_omnibus)
display(voice_any_country_pairwise)
print("Voice-message-rate country moderation:")
display(voice_country_omnibus)
display(voice_country_pairwise)
print("Mode tables are descriptive only:")
display(mode_inference_note)


Voice descriptives:


,country,condition,n_participants,mean_voice_message_rate,sd_voice_message_rate,any_voice_rate,mean_voice_messages,mean_typed_messages
0,china,Avatar,54,0.210185,0.387710,0.259259,1.314815,4.648148
1,china,Chat,54,0.070032,0.201837,0.185185,0.314815,5.055556
2,germany,Avatar,43,0.467499,0.493036,0.511628,4.093023,4.465116
3,germany,Chat,43,0.000000,0.000000,0.000000,0.000000,8.325581
4,israel,Avatar,49,0.863338,0.336248,0.877551,7.326531,1.163265
5,israel,Chat,49,0.131468,0.296071,0.224490,1.000000,8.224490


Overall paired McNemar:


,outcome,n_pairs,chat_no_avatar_yes,chat_yes_avatar_no,discordant_pairs,mcnemar_statistic_continuity_corrected,p_value
0,used_voice,146,60,2,62,52.403226,8.474124e-16


Within-country paired McNemar with Holm across three countries:


,country,outcome,n_pairs,chat_no_avatar_yes,chat_yes_avatar_no,discordant_pairs,mcnemar_statistic_continuity_corrected,p_value,p_holm_across_countries
0,israel,used_voice,49,32,0,32,30.031250,4.656613e-10,1.396984e-09
1,germany,used_voice,43,22,0,22,20.045455,4.768372e-07,9.536743e-07
2,china,used_voice,54,6,2,8,1.125000,2.890625e-01,2.890625e-01


... (4 outputs left out)

,outcome,analysis_design,n_participants,n_israel,n_germany,n_china_group,wald_chi2,df,p_value,robust_model_r_squared,p_holm
0,voice_message_rate,within_participant_difference_score,146,49,43,54,65.067079,2,7.427852e-15,0.274308,7.427852e-15


,outcome,analysis_design,country_a,country_b,n_a,n_b,mean_delta_a,mean_delta_b,difference_of_deltas_a_minus_b,ci95_low,...,welch_df,p_value,cohens_d,hedges_g,mann_whitney_u,mann_whitney_p,p_holm,mann_whitney_p_holm,omnibus_p_holm,posthoc_interpretation_enabled
0,voice_message_rate,within_participant_difference_score,israel,germany,49,43,0.731870,0.467499,0.264371,0.074999,...,81.872168,6.794616e-03,0.587471,0.582562,1356.5,8.708107e-03,6.794616e-03,8.708107e-03,7.427852e-15,True
1,voice_message_rate,within_participant_difference_score,israel,china,49,54,0.731870,0.140153,0.591718,0.445594,...,92.188278,2.906220e-12,1.603430,1.591493,2227.5,1.026266e-10,8.718660e-12,3.078799e-10,7.427852e-15,True
2,voice_message_rate,within_participant_difference_score,germany,china,43,54,0.467499,0.140153,0.327346,0.152817,...,70.087455,3.718816e-04,0.798956,0.792632,1586.0,3.380603e-04,7.437632e-04,6.761207e-04,7.427852e-15,True


Mode tables are descriptive only:


,test,performed,reason
0,ordinary_chi_square_condition_or_country_by_in...,False,Not used for inference because participant obs...


### Profile of modality switchers

Describe participants who used voice in Avatar but not in Chat and compare their outcome profiles.


In [26]:
# ============================================================
# Exploratory profile of modality switchers:
# Voice in Avatar, but text-only in Chat
# ============================================================

# ------------------------------------------------------------
# Build one row per participant with condition-specific metrics
# ------------------------------------------------------------

voice_profile = (
    analysis[
        [
            "user_id", "session_id", "country", "condition",
            "used_voice", "input_mode_pattern",
            "voice_message_rate",
            "character_connection",
            "satisfaction",
            "knowledge_accuracy",
        ]
    ]
    .copy()
)

# Any voice in Avatar, no voice at all in Chat
used_voice_wide = (
    voice_profile
    .pivot_table(
        index=["user_id", "session_id", "country"],
        columns="condition",
        values="used_voice",
        aggfunc="first"
    )
    .reset_index()
)

used_voice_wide["voice_avatar_text_chat"] = (
    used_voice_wide["Avatar"].astype(bool)
    & ~used_voice_wide["Chat"].astype(bool)
)

print("ANY VOICE IN AVATAR + NO VOICE IN CHAT")
print(
    used_voice_wide["voice_avatar_text_chat"]
    .value_counts()
)

print("\nBy country:")
display(
    used_voice_wide
    .groupby("country")["voice_avatar_text_chat"]
    .agg(["sum", "count", "mean"])
    .assign(percent=lambda d: 100*d["mean"])
    .reset_index()
)


# ------------------------------------------------------------
# Strict version:
# voice-only Avatar + typed-only Chat
# ------------------------------------------------------------

mode_wide = (
    voice_profile
    .pivot_table(
        index=["user_id", "session_id", "country"],
        columns="condition",
        values="input_mode_pattern",
        aggfunc="first"
    )
    .reset_index()
)

mode_wide["strict_switcher"] = (
    (mode_wide["Avatar"] == "voice_only")
    & (mode_wide["Chat"] == "typed_only")
)

print("\nSTRICT: VOICE-ONLY AVATAR + TYPED-ONLY CHAT")
print(mode_wide["strict_switcher"].value_counts())

print("\nStrict switchers by country:")
display(
    mode_wide
    .groupby("country")["strict_switcher"]
    .agg(["sum", "count", "mean"])
    .assign(percent=lambda d: 100*d["mean"])
    .reset_index()
)


# ------------------------------------------------------------
# Avatar-minus-Chat outcomes for each participant
# Importance is intentionally excluded here because it is a global attitude
# and the second interaction may contain carryover from the first.
# ------------------------------------------------------------

outcomes = [
    "character_connection",
    "satisfaction",
    "knowledge_accuracy",
]

participant_deltas = used_voice_wide[
    ["user_id", "session_id", "country", "voice_avatar_text_chat"]
].copy()

for outcome in outcomes:

    temp = (
        analysis
        .pivot_table(
            index=["user_id", "session_id"],
            columns="condition",
            values=outcome,
            aggfunc="mean"
        )
        .reset_index()
    )

    temp[f"{outcome}_delta"] = (
        temp["Avatar"] - temp["Chat"]
    )

    participant_deltas = participant_deltas.merge(
        temp[
            ["user_id", "session_id", f"{outcome}_delta"]
        ],
        on=["user_id", "session_id"],
        how="left"
    )


# ------------------------------------------------------------
# Descriptive comparison:
# switchers vs everyone else
# ------------------------------------------------------------

delta_cols = [f"{x}_delta" for x in outcomes]

switcher_summary = (
    participant_deltas
    .groupby("voice_avatar_text_chat")[delta_cols]
    .agg(["count", "mean", "std"])
)

print("\nAvatar-minus-Chat outcomes:")
display(switcher_summary)


# ------------------------------------------------------------
# Exploratory Welch comparisons:
# Do switchers show larger Avatar advantages?
# ------------------------------------------------------------

results = []

for outcome in outcomes:

    col = f"{outcome}_delta"

    switchers = participant_deltas.loc[
        participant_deltas["voice_avatar_text_chat"], col
    ].dropna()

    others = participant_deltas.loc[
        ~participant_deltas["voice_avatar_text_chat"], col
    ].dropna()

    t, p = stats.ttest_ind(
        switchers,
        others,
        equal_var=False
    )

    results.append({
        "outcome": outcome,
        "n_switchers": len(switchers),
        "n_others": len(others),
        "switcher_mean_delta": switchers.mean(),
        "other_mean_delta": others.mean(),
        "difference": switchers.mean() - others.mean(),
        "welch_t": t,
        "p_value": p,
    })

switcher_tests = pd.DataFrame(results)

switcher_tests["p_holm"] = multipletests(
    switcher_tests["p_value"],
    method="holm"
)[1]

print("\nExploratory switcher vs non-switcher comparisons:")
display(switcher_tests)


ANY VOICE IN AVATAR + NO VOICE IN CHAT
voice_avatar_text_chat
False    86
True     60
Name: count, dtype: int64

By country:


,country,sum,count,mean,percent
0,china,6,54,0.111111,11.111111
1,germany,22,43,0.511628,51.162791
2,israel,32,49,0.653061,65.306122



STRICT: VOICE-ONLY AVATAR + TYPED-ONLY CHAT
strict_switcher
False    93
True     53
Name: count, dtype: int64

Strict switchers by country:


,country,sum,count,mean,percent
0,china,4,54,0.074074,7.407407
1,germany,18,43,0.418605,41.860465
2,israel,31,49,0.632653,63.265306



Avatar-minus-Chat outcomes:


condition              character_connection_delta                      \
                                            count      mean       std   
voice_avatar_text_chat                                                  
False                                          86  0.156977  0.614825   
True                                           60  0.558333  0.892965   

condition              satisfaction_delta                      \
                                    count      mean       std   
voice_avatar_text_chat                                          
False                                  86  0.271318  0.906781   
True                                   60  0.488889  0.894568   

condition              knowledge_accuracy_delta                      
                                          count      mean       std  
voice_avatar_text_chat                                               
False                                        86 -0.004651  0.277767  
True                                         60  0.056667  0.283062


Exploratory switcher vs non-switcher comparisons:


,outcome,n_switchers,n_others,switcher_mean_delta,other_mean_delta,difference,welch_t,p_value,p_holm
0,character_connection,60,86,0.558333,0.156977,0.401357,3.018039,0.003249,0.009748
1,satisfaction,60,86,0.488889,0.271318,0.217571,1.437794,0.152929,0.305859
2,knowledge_accuracy,60,86,0.056667,-0.004651,0.061318,1.297735,0.196757,0.305859


### Does the switcher–connection association differ by country?

Test whether the observed switcher–connection association varies across cohorts.


In [27]:
# ============================================================
# Does the switcher–Connection relationship differ by country?
# ============================================================

connection_country = participant_deltas[
    [
        "user_id",
        "session_id",
        "country",
        "voice_avatar_text_chat",
        "character_connection_delta",
    ]
].dropna().copy()

# ------------------------------------------------------------
# Descriptives within each country
# ------------------------------------------------------------

country_switcher_connection = (
    connection_country
    .groupby(
        ["country", "voice_avatar_text_chat"],
        observed=True
    )
    .agg(
        n=("character_connection_delta", "size"),
        mean_connection_delta=("character_connection_delta", "mean"),
        sd=("character_connection_delta", "std"),
    )
    .reset_index()
)

print("Connection Avatar-minus-Chat by switcher status and country:")
display(country_switcher_connection)


# ------------------------------------------------------------
# Switchers vs non-switchers separately within each country
# ------------------------------------------------------------

country_tests = []

for country in sorted(connection_country["country"].unique()):

    temp = connection_country[
        connection_country["country"] == country
    ]

    switchers = temp.loc[
        temp["voice_avatar_text_chat"],
        "character_connection_delta"
    ]

    others = temp.loc[
        ~temp["voice_avatar_text_chat"],
        "character_connection_delta"
    ]

    t, p = stats.ttest_ind(
        switchers,
        others,
        equal_var=False
    )

    country_tests.append({
        "country": country,
        "n_switchers": len(switchers),
        "n_others": len(others),
        "switcher_mean_delta": switchers.mean(),
        "other_mean_delta": others.mean(),
        "difference": switchers.mean() - others.mean(),
        "welch_t": t,
        "p_value": p,
    })

country_tests = pd.DataFrame(country_tests)

country_tests["p_holm"] = multipletests(
    country_tests["p_value"],
    method="holm"
)[1]

print("\nWithin-country switcher vs non-switcher comparisons:")
display(country_tests)


# ------------------------------------------------------------
# Formal interaction:
# Does the switcher effect differ across countries?
# ------------------------------------------------------------

connection_country["switcher"] = (
    connection_country["voice_avatar_text_chat"].astype(int)
)

model = smf.ols(
    "character_connection_delta ~ switcher * C(country)",
    data=connection_country
).fit(cov_type="HC3")

interaction_terms = [
    term for term in model.params.index
    if "switcher:C(country)" in term
]

wald_stat, df, p_interaction = _joint_wald(
    model,
    interaction_terms
)

print(
    f"\nSwitcher × Country interaction for Connection: "
    f"Wald chi2({df})={wald_stat:.3f}, "
    f"p={p_interaction:.4f}"
)


Connection Avatar-minus-Chat by switcher status and country:


,country,voice_avatar_text_chat,n,mean_connection_delta,sd
0,china,False,48,0.135417,0.513018
1,china,True,6,-0.166667,0.408248
2,germany,False,21,0.261905,0.717967
3,germany,True,22,0.568182,0.942331
4,israel,False,17,0.088235,0.754886
5,israel,True,32,0.687500,0.877588



Within-country switcher vs non-switcher comparisons:


,country,n_switchers,n_others,switcher_mean_delta,other_mean_delta,difference,welch_t,p_value,p_holm
0,china,6,48,-0.166667,0.135417,-0.302083,-1.656381,0.140779,0.281558
1,germany,22,21,0.568182,0.261905,0.306277,1.202154,0.236531,0.281558
2,israel,32,17,0.687500,0.088235,0.599265,2.497190,0.017056,0.051169



Switcher × Country interaction for Connection: Wald chi2(2)=8.884, p=0.0118


# Part VII - Secondary Behavioral Country Heterogeneity

Apply cohort-heterogeneity tests to the full 17-outcome secondary behavioral family.


In [28]:
country_omnibus_secondary = country_moderation_omnibus(
    participant_differences, SECONDARY_BEHAVIOR_OUTCOMES, MIN_GROUP_N
)
country_pairwise_secondary = country_pairwise_posthoc(
    participant_differences,
    SECONDARY_BEHAVIOR_OUTCOMES,
    country_omnibus_secondary,
    MIN_GROUP_N,
)

print("SECONDARY OMNIBUS COUNTRY MODERATION")
display(country_omnibus_secondary.sort_values("p_holm", na_position="last"))

print("SECONDARY POST-HOC COUNTRY PAIRS")
display(country_pairwise_secondary.sort_values(["outcome", "p_holm"], na_position="last"))


SECONDARY OMNIBUS COUNTRY MODERATION


,outcome,analysis_design,n_participants,n_israel,n_germany,n_china_group,wald_chi2,df,p_value,robust_model_r_squared,p_holm
10,n_voice_messages,within_participant_difference_score,146,49,43,54,71.665887,2,2.741261e-16,0.282591,4.660144e-15
9,voice_message_rate,within_participant_difference_score,146,49,43,54,65.067079,2,7.427852e-15,0.274308,1.188456e-13
11,n_typed_messages,within_participant_difference_score,146,49,43,54,50.151213,2,1.287664e-11,0.231638,1.931496e-10
12,n_input_switches,within_participant_difference_score,146,49,43,54,5.374364,2,6.807250e-02,0.021016,9.530150e-01
0,n_messages,within_participant_difference_score,146,49,43,54,2.558221,2,2.782847e-01,0.018366,1.000000e+00
14,mean_agent_response_latency_seconds,within_participant_difference_score,146,49,43,54,2.685267,2,2.611570e-01,0.025008,1.000000e+00
13,mean_agent_words_per_response,within_participant_difference_score,146,49,43,54,2.409739,2,2.997311e-01,0.020110,1.000000e+00
15,agent_response_coverage,within_participant_difference_score,146,49,43,54,0.458865,2,4.981549e-01,0.006442,1.000000e+00
8,messages_per_minute,within_participant_difference_score,145,48,43,54,0.142384,2,9.312832e-01,0.000818,1.000000e+00
6,question_form_rate,within_participant_difference_score,146,49,43,54,2.539961,2,2.808371e-01,0.021648,1.000000e+00


SECONDARY POST-HOC COUNTRY PAIRS


,outcome,analysis_design,country_a,country_b,n_a,n_b,mean_delta_a,mean_delta_b,difference_of_deltas_a_minus_b,ci95_low,...,welch_df,p_value,cohens_d,hedges_g,mann_whitney_u,mann_whitney_p,p_holm,mann_whitney_p_holm,omnibus_p_holm,posthoc_interpretation_enabled
45,agent_response_coverage,within_participant_difference_score,israel,germany,49,43,0.003817,0.000000,0.003817,-0.007397,...,48.000000,4.970042e-01,0.133882,0.132763,1075.0,5.932819e-01,9.940085e-01,1.000000e+00,1.000000e+00,False
46,agent_response_coverage,within_participant_difference_score,israel,china,49,54,0.003817,0.000000,0.003817,-0.007397,...,48.000000,4.970042e-01,0.141828,0.140772,1350.0,5.480338e-01,9.940085e-01,1.000000e+00,1.000000e+00,False
47,agent_response_coverage,within_participant_difference_score,germany,china,43,54,0.000000,0.000000,0.000000,NaN,...,NaN,NaN,NaN,NaN,1161.0,1.000000e+00,NaN,1.000000e+00,1.000000e+00,False
12,conversation_type_token_ratio,within_participant_difference_score,israel,germany,49,43,0.009763,-0.002242,0.012005,-0.040409,...,89.716190,6.501762e-01,0.093924,0.093139,1122.0,5.946291e-01,1.000000e+00,1.000000e+00,1.000000e+00,False
13,conversation_type_token_ratio,within_participant_difference_score,israel,china,49,53,0.009763,-0.000125,0.009888,-0.042898,...,97.952189,7.108920e-01,0.073863,0.073308,1318.0,8.987349e-01,1.000000e+00,1.000000e+00,1.000000e+00,False
14,conversation_type_token_ratio,within_participant_difference_score,germany,china,43,53,-0.002242,-0.000125,-0.002117,-0.051643,...,93.272196,9.325442e-01,-0.017196,-0.017059,1096.0,7.513858e-01,1.000000e+00,1.000000e+00,1.000000e+00,False
21,interaction_duration_minutes,within_participant_difference_score,israel,germany,49,43,-0.591758,0.492464,-1.084222,-2.398146,...,79.143677,1.044687e-01,-0.332342,-0.329565,904.0,2.436095e-01,2.854325e-01,4.872189e-01,1.000000e+00,False
22,interaction_duration_minutes,within_participant_difference_score,israel,china,49,54,-0.591758,0.464467,-1.056225,-2.301396,...,70.645974,9.514417e-02,-0.343256,-0.340701,1051.0,7.299367e-02,2.854325e-01,2.189810e-01,1.000000e+00,False
23,interaction_duration_minutes,within_participant_difference_score,germany,china,43,54,0.492464,0.464467,0.027997,-0.860807,...,84.753670,9.502069e-01,0.012977,0.012875,1028.0,3.359525e-01,9.502069e-01,4.872189e-01,1.000000e+00,False
42,mean_agent_response_latency_seconds,within_participant_difference_score,israel,germany,49,43,-1.256584,0.014178,-1.270762,-3.204346,...,48.044149,1.926366e-01,-0.258494,-0.256334,889.0,1.993533e-01,5.082836e-01,5.980600e-01,1.000000e+00,False


# Part VIII - Exploratory Moderator Sensitivity

Run the exploratory moderator families for gender, character, and presentation order.

In [29]:
def fit_exploratory_moderator(data, outcomes, moderator, continuous=False):
    coef_frames, target_rows = [], []
    for outcome in outcomes:
        if outcome not in data.columns or moderator not in data.columns:
            continue
        if outcome == "importance_change" and moderator == "condition_position":
            # Once importance is restricted to interaction 1, condition_position is constant (=1),
            # so an order moderation test is not defined for this outcome.
            continue

        frame = data.copy()
        if outcome == "importance_change":
            frame = frame.loc[frame["condition_order"].eq("first")].copy()

        frame[outcome] = pd.to_numeric(frame[outcome], errors="coerce")
        required = [outcome, "condition", moderator, "user_id"]
        frame = frame.dropna(subset=required)
        if len(frame) < 20 or frame["condition"].nunique() < 2 or frame[moderator].nunique() < 2:
            continue

        if continuous:
            centered = f"{moderator}_centered"
            numeric_mod = pd.to_numeric(frame[moderator], errors="coerce")
            frame[centered] = numeric_mod - numeric_mod.mean()
            mod_term = centered
        else:
            mod_term = f"C({moderator})"

        terms = [f"C(condition) * {mod_term}"]
        if moderator != "character" and "character" in frame.columns:
            terms.append("C(character)")
        if moderator != "country" and "country" in frame.columns and frame["country"].nunique() >= 2:
            terms.append("C(country)")
        if (
            outcome != "importance_change"
            and moderator != "condition_position"
            and "condition_position" in frame.columns
            and frame["condition_position"].notna().sum() >= 10
        ):
            terms.append("condition_position")
        formula = f'Q("{outcome}") ~ ' + " + ".join(terms)

        if outcome == "importance_change":
            fit = smf.ols(formula, frame).fit(cov_type="HC3")
            model_kind = "first_interaction_HC3_OLS_exploratory"
        else:
            fit = smf.ols(formula, frame).fit(
                cov_type="cluster", cov_kwds={"groups": frame["user_id"]}
            )
            model_kind = "participant_clustered_OLS_exploratory"

        coef = hoa.flatten_model(fit, f"{model_kind}: {formula}", outcome)
        coef["moderator"] = moderator
        coef["model_kind"] = model_kind
        coef_frames.append(coef)

        target_terms = [
            n for n in fit.params.index
            if ":" in n and "C(condition)" in n and (
                (continuous and f"{moderator}_centered" in n)
                or ((not continuous) and f"C({moderator})" in n)
            )
        ]
        stat, df, p = _joint_wald(fit, target_terms)
        target_rows.append({
            "outcome": outcome,
            "moderator": moderator,
            "analysis_design": (
                "first_interaction_between_participants"
                if outcome == "importance_change"
                else "repeated_measures_condition_interaction"
            ),
            "interaction_df": df,
            "wald_chi2": stat,
            "p_value": p,
        })

    coefs = pd.concat(coef_frames, ignore_index=True) if coef_frames else pd.DataFrame()
    targets = pd.DataFrame(target_rows)
    if not targets.empty:
        targets["p_holm"] = _holm(targets["p_value"])
    return coefs, targets


character_mod_coefs, character_mod_interactions = fit_exploratory_moderator(
    analysis, PRIMARY_OUTCOMES, "character", continuous=False
)
order_mod_coefs, order_mod_interactions = fit_exploratory_moderator(
    analysis, PRIMARY_OUTCOMES, "condition_position", continuous=True
)

binary_gender_analysis = analysis.loc[
    analysis["gender_normalized"].isin(["female", "male"])
].copy()
gender_mod_coefs, gender_mod_interactions = fit_exploratory_moderator(
    binary_gender_analysis,
    PRIMARY_OUTCOMES,
    "gender_normalized",
    continuous=False,
)

gender_composition = (
    analysis[["user_id", "session_id", "country", "gender_normalized"]]
    .drop_duplicates(["user_id", "session_id"])
    .groupby(["country", "gender_normalized"], dropna=False)
    .size()
    .reset_index(name="n_participants")
)

print("Character moderation - targeted interactions:")
display(character_mod_interactions)
print("Presentation-order moderation - targeted interactions (importance omitted because first-only):")
display(order_mod_interactions)
print("Gender moderation - targeted interactions:")
display(gender_mod_interactions)
print("Gender composition by research group:")
display(gender_composition)


Character moderation - targeted interactions:


,outcome,moderator,analysis_design,interaction_df,wald_chi2,p_value,p_holm
0,character_connection,character,repeated_measures_condition_interaction,1,4.407498,0.035781,0.143125
1,satisfaction,character,repeated_measures_condition_interaction,1,2.464467,0.116447,0.349342
2,knowledge_accuracy,character,repeated_measures_condition_interaction,1,1.841157,0.174815,0.349631
3,importance_change,character,first_interaction_between_participants,1,0.297010,0.585763,0.585763


Presentation-order moderation - targeted interactions (importance omitted because first-only):


,outcome,moderator,analysis_design,interaction_df,wald_chi2,p_value,p_holm
0,character_connection,condition_position,repeated_measures_condition_interaction,1,1.636790,0.200767,0.528548
1,satisfaction,condition_position,repeated_measures_condition_interaction,1,0.556942,0.455495,0.528548
2,knowledge_accuracy,condition_position,repeated_measures_condition_interaction,1,1.829535,0.176183,0.528548


Gender moderation - targeted interactions:


,outcome,moderator,analysis_design,interaction_df,wald_chi2,p_value,p_holm
0,character_connection,gender_normalized,repeated_measures_condition_interaction,1,0.002157,0.962959,1.0
1,satisfaction,gender_normalized,repeated_measures_condition_interaction,1,0.349005,0.554677,1.0
2,knowledge_accuracy,gender_normalized,repeated_measures_condition_interaction,1,0.505911,0.476914,1.0
3,importance_change,gender_normalized,first_interaction_between_participants,1,0.044320,0.833259,1.0


Gender composition by research group:


,country,gender_normalized,n_participants
0,china,female,13
1,china,male,41
2,germany,female,7
3,germany,male,35
4,germany,other,1
5,israel,female,13
6,israel,male,35
7,israel,other,1


# Part IX - Design-Sensitivity Benchmark

Calculate the paired standardized effect detectable with 80% power under the stated multiplicity benchmark.


In [30]:
from statsmodels.stats.power import TTestPower, TTestIndPower

primary_alpha_benchmark = 0.05 / len(PRIMARY_OUTCOMES)

paired_mde_abs_dz = TTestPower().solve_power(
    nobs=final_sample_n,
    alpha=primary_alpha_benchmark,
    power=0.80,
    alternative="two-sided",
)

n_avatar_first = int((importance_first["condition"] == "Avatar").sum())
n_chat_first = int((importance_first["condition"] == "Chat").sum())
independent_mde_abs_d = TTestIndPower().solve_power(
    nobs1=n_avatar_first,
    ratio=n_chat_first / n_avatar_first,
    alpha=primary_alpha_benchmark,
    power=0.80,
    alternative="two-sided",
)

primary_power_sensitivity = pd.DataFrame([
    {
        "outcomes": ", ".join(PAIRED_PRIMARY_OUTCOMES),
        "analysis_design": "paired",
        "n_group_1": int(final_sample_n),
        "n_group_2": int(final_sample_n),
        "two_sided_alpha_benchmark": float(primary_alpha_benchmark),
        "target_power": 0.80,
        "minimum_detectable_abs_effect": float(paired_mde_abs_dz),
        "effect_metric": "Cohen's dz",
        "interpretation": "conservative paired-test sensitivity benchmark; not an equivalence bound",
    },
    {
        "outcomes": "importance_change",
        "analysis_design": "between_participants_first_interaction_only",
        "n_group_1": n_avatar_first,
        "n_group_2": n_chat_first,
        "two_sided_alpha_benchmark": float(primary_alpha_benchmark),
        "target_power": 0.80,
        "minimum_detectable_abs_effect": float(independent_mde_abs_d),
        "effect_metric": "Cohen's d",
        "interpretation": "conservative independent-groups first-interaction sensitivity benchmark; not an equivalence bound",
    },
])

primary_rank_sensitivity = whole_sample_primary[[
    "outcome", "analysis_design", "p_value", "p_holm",
    "rank_test", "rank_test_p", "rank_test_p_holm",
]].copy()

primary_wilcoxon_sensitivity = primary_rank_sensitivity.copy()

print("Primary mixed-design sensitivity benchmarks:")
display(primary_power_sensitivity)

print("Primary rank-based sensitivity table:")
display(primary_rank_sensitivity)


Primary mixed-design sensitivity benchmarks:


,outcomes,analysis_design,n_group_1,n_group_2,two_sided_alpha_benchmark,target_power,minimum_detectable_abs_effect,effect_metric,interpretation
0,"character_connection, satisfaction, knowledge_...",paired,146,146,0.0125,0.8,0.279377,Cohen's dz,conservative paired-test sensitivity benchmark...
1,importance_change,between_participants_first_interaction_only,73,73,0.0125,0.8,0.558798,Cohen's d,conservative independent-groups first-interact...


Primary rank-based sensitivity table:


,outcome,analysis_design,p_value,p_holm,rank_test,rank_test_p,rank_test_p_holm
0,character_connection,within_participant_paired,0.000001,0.000004,Wilcoxon signed-rank,0.000001,0.000006
1,satisfaction,within_participant_paired,0.000004,0.000011,Wilcoxon signed-rank,0.000006,0.000017
2,knowledge_accuracy,within_participant_paired,0.377749,0.755498,Wilcoxon signed-rank,0.393409,0.786817
3,importance_change,between_participants_first_interaction_only,0.613305,0.755498,Mann-Whitney U,0.689514,0.786817


# Part X - Recruitment Timing and Model-Version Audit

Summarize recruitment timing and reconstruct the documented model-version exposure by participant.


In [31]:
participant_first_event = (
    conversation_events.groupby(
        ["user_id", "session_id", "country"], as_index=False
    )["client_timestamp"].min()
)
participant_first_event["client_timestamp"] = pd.to_datetime(
    participant_first_event["client_timestamp"],
    utc=True,
    errors="coerce",
)

recruitment_timing_by_country = (
    participant_first_event
    .groupby("country")["client_timestamp"]
    .agg(
        first_retained_session="min",
        last_retained_session="max",
        n_participants="count",
    )
    .reset_index()
)

# Documented server-default cutovers used for the deployment audit.
local_event = participant_first_event.copy()
local_event["client_timestamp_israel"] = (
    local_event["client_timestamp"].dt.tz_convert("Asia/Jerusalem")
)

cutover_20_end = pd.Timestamp(
    "2026-06-05 13:40:00", tz="Asia/Jerusalem"
)
cutover_25_end = pd.Timestamp(
    "2026-06-05 13:58:00", tz="Asia/Jerusalem"
)

local_event["derived_model_version"] = np.select(
    [
        local_event["client_timestamp_israel"] < cutover_20_end,
        local_event["client_timestamp_israel"] < cutover_25_end,
    ],
    [
        "gemini-2.0-flash-001",
        "gemini-2.5-flash",
    ],
    default="gemini-3.1-flash-lite",
)

derived_model_version_site_audit = (
    local_event
    .groupby(["derived_model_version", "country"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

derived_model_version_site_audit["provenance"] = (
    "Derived from first retained conversation timestamp and documented "
    "server-default cutovers; model version was not directly logged per interaction."
)

print("Retained participant collection windows by research group:")
display(recruitment_timing_by_country)

print("Derived model-version × research-group audit:")
display(derived_model_version_site_audit)


Retained participant collection windows by research group:


,country,first_retained_session,last_retained_session,n_participants
0,china,2026-08-21 12:44:46.910000+00:00,2026-08-28 15:32:29.252000+00:00,54
1,germany,2026-06-21 17:40:39.688000+00:00,2026-07-07 18:21:27.718000+00:00,43
2,israel,2026-04-09 17:17:13.514000+00:00,2026-09-03 19:13:25.348000+00:00,49


Derived model-version × research-group audit:


country,derived_model_version,china,germany,israel,provenance
0,gemini-2.0-flash-001,0,0,40,Derived from first retained conversation times...
1,gemini-3.1-flash-lite,54,43,9,Derived from first retained conversation times...


# Part XI - Manuscript-Facing Figures

Generate the figures referenced in the manuscript from the final analysis dataset.


In [ ]:
# ============================================================
# Publication figures used in the manuscript
# ============================================================
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(2026)

GROUP_DISPLAY = {
    "israel": "Israel",
    "germany": "Germany",
    "china": "China",
}


# ============================================================
# Figure 1 - manuscript main figure:
# first three outcomes show participant-level Avatar-minus-Chat paired differences;
# importance shows baseline-to-post change after the FIRST interaction only,
# split by first assigned condition (Avatar-first vs Chat-first).
# ============================================================

PRIMARY_FIGURE_OUTCOMES = [
    ("character_connection", "Character connection"),
    ("satisfaction", "Satisfaction"),
    ("knowledge_accuracy", "Knowledge accuracy"),
    ("importance_change", "Importance change\n(first interaction only)"),
]

fig, axes = plt.subplots(2, 2, figsize=(10, 6.5), constrained_layout=True)
axes = axes.flatten()
primary_figure_rows = []

for ax, (outcome, label) in zip(axes, PRIMARY_FIGURE_OUTCOMES):
    stat_row = whole_sample_primary.loc[whole_sample_primary["outcome"].eq(outcome)]
    if len(stat_row) != 1:
        raise AssertionError(f"Expected one whole-sample result row for {outcome}; got {len(stat_row)}.")
    stat_row = stat_row.iloc[0]

    if outcome != "importance_change":
        wide = analysis.pivot_table(
            index=["user_id", "session_id"], columns="condition", values=outcome, aggfunc="mean"
        )
        paired = wide[["Chat", "Avatar"]].apply(pd.to_numeric, errors="coerce").dropna()
        diff = paired["Avatar"] - paired["Chat"]
        mean_diff = float(diff.mean())
        ci_low, ci_high = map(float, hoa.ci_mean(diff))
        assert len(diff) == int(stat_row["n_pairs"])
        assert np.isclose(mean_diff, float(stat_row["mean_difference_avatar_minus_chat"]), atol=1e-10)
        assert np.isclose(ci_low, float(stat_row["difference_ci95_low"]), atol=1e-10)
        assert np.isclose(ci_high, float(stat_row["difference_ci95_high"]), atol=1e-10)

        jitter = rng.uniform(-0.14, 0.14, size=len(diff))
        ax.scatter(diff, jitter, s=24, alpha=0.40)
        ax.errorbar(
            mean_diff, 0.30,
            xerr=[[mean_diff-ci_low], [ci_high-mean_diff]],
            fmt="o", markersize=8, linewidth=2, capsize=5,
        )
        ax.axvline(0, linestyle="--", linewidth=1)
        ax.set_xlabel("Avatar - Chat difference")
        ax.set_yticks([])
        ax.set_ylim(-0.22, 0.43)
        ax.text(
            0.02, 0.96,
            f"Mean = {mean_diff:.3f}\n95% CI [{ci_low:.3f}, {ci_high:.3f}]",
            transform=ax.transAxes, ha="left", va="top", fontsize=9,
        )
        primary_figure_rows.append({
            "outcome": outcome,
            "analysis_design": "within_participant_paired",
            "n": len(diff),
            "mean_difference": mean_diff,
            "ci95_low": ci_low,
            "ci95_high": ci_high,
        })
    else:
        frame = importance_first[["condition", "importance_change"]].dropna().copy()
        group_rows = []
        for y, condition in zip([0.10, -0.10], ["Avatar", "Chat"]):
            vals = frame.loc[frame["condition"].eq(condition), "importance_change"].astype(float)
            mean = float(vals.mean())
            lo, hi = map(float, hoa.ci_mean(vals))
            y_jitter = np.full(len(vals), y) + rng.uniform(-0.035, 0.035, size=len(vals))
            ax.scatter(vals, y_jitter, s=22, alpha=0.38, label=f"{condition}-first")
            ax.errorbar(
                mean, y,
                xerr=[[mean-lo], [hi-mean]],
                fmt="o", markersize=8, linewidth=2, capsize=5,
            )
            group_rows.append((condition, len(vals), mean, lo, hi))
        ax.axvline(0, linestyle="--", linewidth=1)
        ax.set_xlabel("Post-1 - baseline importance")
        ax.set_yticks([0.10, -0.10], ["Avatar-first", "Chat-first"])
        ax.set_ylim(-0.22, 0.22)
        diff = float(stat_row["mean_difference_avatar_minus_chat"])
        ci_low = float(stat_row["difference_ci95_low"])
        ci_high = float(stat_row["difference_ci95_high"])
        ax.text(
            0.02, 0.96,
            f"Between-group Δ = {diff:.3f}\n95% CI [{ci_low:.3f}, {ci_high:.3f}]",
            transform=ax.transAxes, ha="left", va="top", fontsize=9,
        )
        assert int(stat_row["n_avatar"]) == int((frame["condition"] == "Avatar").sum())
        assert int(stat_row["n_chat"]) == int((frame["condition"] == "Chat").sum())
        primary_figure_rows.append({
            "outcome": outcome,
            "analysis_design": "between_participants_first_interaction_only",
            "n": len(frame),
            "mean_difference": diff,
            "ci95_low": ci_low,
            "ci95_high": ci_high,
        })

    ax.set_title(label)

primary_figure_summary = pd.DataFrame(primary_figure_rows)
primary_figure_path = FIGURE_DIR / "primary_difference_distributions.png"
if primary_figure_path.exists():
    primary_figure_path.unlink()
fig.savefig(primary_figure_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

print("✓ Primary-outcome manuscript figure regenerated with first-interaction-only importance:")
print(primary_figure_path)
display(primary_figure_summary)


# ============================================================
# Figure 2 - manuscript main figure:
# participant-level Avatar-minus-Chat voice-message-rate
# differences across the THREE research/deployment cohorts.
# ============================================================

voice_diff_col = "voice_message_rate_avatar_minus_chat"
if voice_diff_col not in participant_differences.columns:
    raise KeyError(
        f"{voice_diff_col!r} was not found in participant_differences."
    )

voice_plot = participant_differences[
    ["country", voice_diff_col]
].copy()
voice_plot[voice_diff_col] = pd.to_numeric(
    voice_plot[voice_diff_col], errors="coerce"
)
voice_plot = voice_plot.dropna(subset=["country", voice_diff_col])

fig, ax = plt.subplots(
    figsize=(6.8, 5.3),
    constrained_layout=True,
)

voice_country_rows = []

for x, country in enumerate(COUNTRY_ORDER):
    values = (
        voice_plot.loc[
            voice_plot["country"].eq(country),
            voice_diff_col,
        ]
        .dropna()
        .astype(float)
    )

    if values.empty:
        continue

    mean_diff = float(values.mean())
    ci_low, ci_high = hoa.ci_mean(values)
    ci_low = float(ci_low)
    ci_high = float(ci_high)

    voice_country_rows.append({
        "country_code": country,
        "group_label": GROUP_DISPLAY.get(country, country.title()),
        "n": len(values),
        "mean_difference": mean_diff,
        "ci95_low": ci_low,
        "ci95_high": ci_high,
    })

    # Horizontal jitter only for visibility.
    x_jitter = (
        np.full(len(values), x)
        + rng.uniform(-0.10, 0.10, size=len(values))
    )
    ax.scatter(x_jitter, values, s=28, alpha=0.45)

    ax.errorbar(
        x,
        mean_diff,
        yerr=[
            [mean_diff - ci_low],
            [ci_high - mean_diff],
        ],
        fmt="o",
        markersize=9,
        linewidth=2.2,
        capsize=6,
    )

ax.axhline(0, linestyle="--", linewidth=1)

ax.set_xticks(
    range(len(COUNTRY_ORDER)),
    [GROUP_DISPLAY.get(country, country.title()) for country in COUNTRY_ORDER],
)
ax.set_ylabel("Voice-message-rate difference\n(Avatar - Chat)")
ax.set_xlabel("Research/deployment cohort")
ax.set_ylim(-0.05, 1.05)

voice_country_figure_path = FIGURE_DIR / "voice_rate_difference_by_country.png"
if voice_country_figure_path.exists():
    voice_country_figure_path.unlink()

fig.savefig(voice_country_figure_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

voice_country_summary = pd.DataFrame(voice_country_rows)

# Validate the three plotted sample sizes against the final analytic sample.
_plot_n = {
    row["country_code"]: int(row["n"])
    for row in voice_country_rows
}
_expected_n = (
    analysis[["user_id", "session_id", "country"]]
    .drop_duplicates()
    .groupby("country")
    .size()
    .to_dict()
)
for country in COUNTRY_ORDER:
    if _plot_n.get(country) != int(_expected_n.get(country, -1)):
        raise AssertionError(
            f"Voice figure N mismatch for {country}: "
            f"{_plot_n.get(country)} vs {_expected_n.get(country)}"
        )

print("\n✓ Three-group voice manuscript figure regenerated from the final sample:")
print(voice_country_figure_path)
display(voice_country_summary)


# ============================================================
# Additional manuscript/supplementary three-group effect figures
# ============================================================

def plot_country_delta(outcome, ylabel=None):
    diff_col = f"{outcome}_avatar_minus_chat"
    if diff_col not in participant_differences.columns:
        return None

    frame = participant_differences[["country", diff_col]].copy()
    frame[diff_col] = pd.to_numeric(frame[diff_col], errors="coerce")
    frame = frame.dropna()

    rows = []
    for country in COUNTRY_ORDER:
        s = frame.loc[frame["country"].eq(country), diff_col]
        if len(s) < 2:
            continue
        mean = float(s.mean())
        ci_low, ci_high = hoa.ci_mean(s)
        rows.append(
            (
                country,
                mean,
                mean - float(ci_low),
                float(ci_high) - mean,
                len(s),
            )
        )

    if not rows:
        return None

    fig, ax = plt.subplots(figsize=(7.2, 4.6))
    x = np.arange(len(rows))
    means = [r[1] for r in rows]
    lower = [r[2] for r in rows]
    upper = [r[3] for r in rows]

    ax.errorbar(
        x,
        means,
        yerr=[lower, upper],
        fmt="o",
        capsize=5,
    )
    ax.axhline(0, linestyle="--", linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels([
        f"{GROUP_DISPLAY.get(r[0], r[0].title())}\n(n={r[4]})"
        for r in rows
    ])
    ax.set_ylabel(ylabel or f"Avatar − Chat: {outcome}")
    ax.set_xlabel("Research/deployment cohort")
    ax.set_title(f"Three-group heterogeneity in the Avatar effect: {outcome}")
    fig.tight_layout()

    path = FIGURE_DIR / f"country_delta_{outcome}.png"
    if path.exists():
        path.unlink()
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return path


def plot_first_importance_country_effect():
    frame = importance_first[["country", "condition", "importance_change"]].dropna().copy()
    rows = []
    for country in COUNTRY_ORDER:
        sub = frame.loc[frame["country"].eq(country)]
        avatar = sub.loc[sub["condition"].eq("Avatar"), "importance_change"]
        chat = sub.loc[sub["condition"].eq("Chat"), "importance_change"]
        if len(avatar) < 2 or len(chat) < 2:
            continue
        diff, lo, hi, _ = _welch_difference_ci(avatar, chat)
        rows.append((country, diff, diff-lo, hi-diff, len(avatar), len(chat)))
    if not rows:
        return None
    fig, ax = plt.subplots(figsize=(7.2, 4.6))
    x = np.arange(len(rows))
    ax.errorbar(
        x, [r[1] for r in rows],
        yerr=[[r[2] for r in rows], [r[3] for r in rows]],
        fmt="o", capsize=5,
    )
    ax.axhline(0, linestyle="--", linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels([
        f"{GROUP_DISPLAY.get(r[0], r[0].title())}\n(A={r[4]}, C={r[5]})" for r in rows
    ])
    ax.set_ylabel("Avatar-first − Chat-first\nimportance change after interaction 1")
    ax.set_xlabel("Research/deployment cohort")
    ax.set_title("Cohort heterogeneity in first-interaction importance effect")
    fig.tight_layout()
    path = FIGURE_DIR / "country_delta_importance_change.png"
    if path.exists():
        path.unlink()
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return path


additional_figure_files = []
for outcome in PAIRED_PRIMARY_OUTCOMES + ["voice_message_rate"]:
    path = plot_country_delta(outcome)
    if path is not None:
        additional_figure_files.append(path)
importance_country_figure = plot_first_importance_country_effect()
if importance_country_figure is not None:
    additional_figure_files.append(importance_country_figure)


# ============================================================
# Final figure manifest
# ============================================================

publication_figure_files = [
    primary_figure_path,
    voice_country_figure_path,
] + additional_figure_files

print("\nGenerated/overwritten manuscript figure files:")
for path in publication_figure_files:
    print(" -", path)


# Part XII - Export Manuscript and Audit Tables

Export the manuscript-facing statistical tables, audit files, and derived participant-level outputs.


In [ ]:
# Manuscript-facing multiplicity plan.
def _primary_design_label(outcome):
    return (
        "between_participants_first_interaction_only"
        if outcome == "importance_change"
        else "within_participant_paired"
    )

canonical_multiplicity_families = pd.DataFrame(
    [
        {"family": "primary_whole_sample", "family_size": len(PRIMARY_OUTCOMES), "outcome": o, "correction": "Holm", "analysis_design": _primary_design_label(o)}
        for o in PRIMARY_OUTCOMES
    ] + [
        {"family": "primary_country_omnibus", "family_size": len(PRIMARY_OUTCOMES), "outcome": o, "correction": "Holm", "analysis_design": _primary_design_label(o)}
        for o in PRIMARY_OUTCOMES
    ] + [
        {"family": "secondary_behavior_whole_sample", "family_size": len(SECONDARY_BEHAVIOR_OUTCOMES), "outcome": o, "correction": "Holm"}
        for o in SECONDARY_BEHAVIOR_OUTCOMES
    ] + [
        {"family": "secondary_behavior_country_omnibus", "family_size": len(SECONDARY_BEHAVIOR_OUTCOMES), "outcome": o, "correction": "Holm"}
        for o in SECONDARY_BEHAVIOR_OUTCOMES
    ]
)

country_field_validation = country_audit[[
    "user_id", "country", "country_source"
]].copy()

quality_rows = [
    {"metric": "main_dataset_rows_before_filter", "value": len(study_raw)},
    {"metric": "participants_final", "value": analysis["user_id"].nunique()},
    {"metric": "interaction_rows", "value": len(analysis)},
    {"metric": "clean_messages", "value": len(messages_clean)},
    {"metric": "knowledge_item_rows", "value": len(knowledge_items)},
]
for country, n in analysis[["user_id", "country"]].drop_duplicates().groupby("country")["user_id"].nunique().items():
    quality_rows.append({"metric": f"participants_{country}", "value": int(n)})
quality = pd.DataFrame(quality_rows)


# Input-method provenance summary for reproducibility.
def _provenance_counts(frame, scope_name):
    counts = (
        frame["input_method_source"]
        .fillna("unknown")
        .value_counts(dropna=False)
        .rename_axis("input_method_source")
        .reset_index(name="n_messages")
    )
    counts.insert(0, "scope", scope_name)
    counts["total_messages_in_scope"] = len(frame)
    return counts

input_method_provenance_summary = pd.concat([
    _provenance_counts(
        messages_labeled_unfiltered,
        "all_extracted_user_turns",
    ),
    _provenance_counts(
        messages_labeled,
        "before_greeting_removal",
    ),
    _provenance_counts(
        messages_clean,
        "final_genuine_messages",
    ),
], ignore_index=True)

outputs = {
    # Manuscript-facing primary tables
    "manuscript_primary_whole_sample.csv": whole_sample_primary,
    "manuscript_primary_country_omnibus.csv": country_omnibus_primary,
    "manuscript_primary_country_posthoc.csv": country_pairwise_primary,
    "manuscript_primary_country_descriptives.csv": country_descriptives_primary,
    "manuscript_primary_within_country_descriptive.csv": country_paired_primary,
    "manuscript_primary_adjusted_interaction_tests.csv": primary_adjusted_interactions,
    "supplement_primary_adjusted_coefficients.csv": primary_adjusted_coefficients,
    "manuscript_importance_first_interaction.csv": importance_first_primary,
    "importance_first_interaction_baseline_balance.csv": importance_first_interaction_baseline_balance,
    "importance_first_interaction_country_omnibus.csv": importance_country_omnibus,
    "importance_first_interaction_country_posthoc.csv": importance_country_pairwise,
    "importance_first_interaction_baseline_le4.csv": first_only_room_comparison,

    # Knowledge robustness
    "manuscript_knowledge_item_gee_interaction.csv": knowledge_gee_interaction,
    "supplement_knowledge_item_gee_coefficients.csv": knowledge_gee_coefficients,

    # Voice / behavior
    "manuscript_secondary_whole_sample.csv": whole_sample_secondary,
    "manuscript_secondary_country_omnibus.csv": country_omnibus_secondary,
    "manuscript_secondary_country_posthoc.csv": country_pairwise_secondary,
    "manuscript_voice_descriptives.csv": voice_descriptives,
    "manuscript_voice_mcnemar_overall.csv": mcnemar_voice_overall,
    "manuscript_voice_mcnemar_by_country.csv": mcnemar_voice_by_country,
    "manuscript_voice_any_change_country_omnibus.csv": voice_any_country_omnibus,
    "manuscript_voice_any_change_country_posthoc.csv": voice_any_country_pairwise,
    "manuscript_voice_rate_country_omnibus.csv": voice_country_omnibus,
    "manuscript_voice_rate_country_posthoc.csv": voice_country_pairwise,
    "descriptive_condition_mode_table.csv": condition_mode_table,
    "descriptive_country_condition_mode_table.csv": country_condition_mode_table,
    "mode_inference_note.csv": mode_inference_note,

    # Exploratory moderator tables
    "exploratory_character_interactions.csv": character_mod_interactions,
    "exploratory_order_interactions.csv": order_mod_interactions,
    "supplement_character_coefficients.csv": character_mod_coefs,
    "supplement_order_coefficients.csv": order_mod_coefs,
    "exploratory_gender_interactions.csv": gender_mod_interactions,
    "supplement_gender_coefficients.csv": gender_mod_coefs,
    "supplement_gender_composition.csv": gender_composition,

    # Design-sensitivity outputs
    "primary_power_sensitivity.csv": primary_power_sensitivity,
    "primary_wilcoxon_sensitivity.csv": primary_wilcoxon_sensitivity,

    # Recruitment/model-version audit
    "recruitment_timing_by_country.csv": recruitment_timing_by_country,
    "derived_model_version_site_audit.csv": derived_model_version_site_audit,


    # Scale / sample / audit tables
    "questionnaire_reliability.csv": reliability,
    "canonical_multiplicity_families.csv": canonical_multiplicity_families,
    "country_field_validation.csv": country_field_validation,
    "country_assignments_audit.csv": country_audit,
    "participant_interface_eligibility.csv": participant_interface_eligibility,
    "excluded_incomplete_interfaces.csv": excluded_incomplete_interfaces,
    "analysis_merged_interaction_level.csv": analysis,
    "participant_paired_differences.csv": participant_differences,
    "input_method_provenance_summary.csv": input_method_provenance_summary,
    "messages_clean.csv": messages_clean,
    "messages_removed_audit.csv": messages_removed,
    "knowledge_items.csv": knowledge_items,
    "data_quality_summary.csv": quality,
}

for filename, frame in outputs.items():
    if not isinstance(frame, pd.DataFrame):
        raise TypeError(f"Output {filename!r} is not a DataFrame: {type(frame)!r}")
    frame.to_csv(OUTPUT_DIR / filename, index=False)


print("Saved", len(outputs), "CSV tables to", OUTPUT_DIR)
display(quality)
